<a href="https://colab.research.google.com/github/LysanetsAndriy/Ukrainian-NER/blob/main/NER_MamayLM_9B_GEPA_PROMPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # In terminal or a notebook cell
# !du -sh ~/.cache/huggingface/
# !rm -rf ~/.cache/huggingface/hub/models--INSAIT-Institute--MamayLM-Gemma-3-12B-IT-v1.0
# !rm -rf ~/.cache/huggingface/hub/models--INSAIT-Institute--MamayLM-Gemma-2-9B-IT-v0.1
# !du -sh ~/.cache/huggingface/

11G	/root/.cache/huggingface/
2.1M	/root/.cache/huggingface/


In [ ]:
# !pip install "peft==0.13.2"
# !pip install -q -U trl

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.4 MB/s eta 0:00:00


In [ ]:
! pip install gepa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 27.7 MB/s eta 0:00:00


In [ ]:
! pip install litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 17.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Succe

# Load model

In [ ]:
import os

# Redirect Hugging Face cache to the persistent volume
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"
os.environ["HF_HUB_CACHE"] = "/content/hf_cache/hub"

os.makedirs("/content/hf_cache", exist_ok=True)

In [ ]:
from huggingface_hub import login
login(token="hf_")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "INSAIT-Institute/MamayLM-Gemma-2-9B-IT-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    # attn_implementation="sdpa"
)


print("Model loaded successfully!")


Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Model loaded successfully!


# Load NER-UK dataset

In [ ]:
# Clone the repo
!git clone https://github.com/lang-uk/ner-uk.git

fatal: destination path 'ner-uk' already exists and is not an empty directory.


In [ ]:
import os
from collections import Counter

In [ ]:
split_file = "ner-uk/v2.0/data/dev-test-split.txt"
dev_files = []
test_files = []

current_split = None
with open(split_file) as f:
    for line in f:
        line = line.strip()
        if line.startswith("DEV"):
            current_split = "dev"
            continue
        elif line.startswith("TEST"):
            current_split = "test"
            continue
        if line:
            if current_split == "dev":
                dev_files.append(line)
            elif current_split == "test":
                test_files.append(line)

print(f"Dev set: {len(dev_files)} files")
print(f"Test set: {len(test_files)} files")
print(f"Total: {len(dev_files) + len(test_files)} files")

Dev set: 391 files
Test set: 169 files
Total: 560 files


In [ ]:
import re

def normalize_and_map_offsets(raw_text, raw_entities):
    """
    Normalizes text based on spacing/punctuation rules and recalculates
    entity start/end offsets dynamically.
    """
    text = raw_text

    # Initialize a mapping array where index = old_char_position, value = new_char_position
    # Size is len + 1 to account for entities that end at the very last character
    old_to_new = list(range(len(raw_text) + 1))

    # Define regex rules (pattern, replacement)
    rules = [
        (r' {2,}', ' '),                                   # 1. Multiple spaces to single space
        (r' +([.,:;!?\)\]»])', r'\1'),                     # 2. Remove space before closing punctuation
        (r'([\({\[«]) +', r'\1'),                          # 3. Remove space after opening punctuation
        (r'[–—]', '-'),                                    # 4. Standardize en/em-dashes to hyphen
        (r'(?<=\d) +(-)+ +(?=\d)', r'\1'),                 # 5. Remove spaces around dashes between digits
        (r'(?<=\d) +(?=%)', ''),                           # 6. Remove space between number and % sign
    ]

    for pattern, repl in rules:
        matches = list(re.finditer(pattern, text))

        # Iterate in REVERSE (right-to-left).
        # This guarantees that changing length at index N doesn't break offsets < N.
        for match in reversed(matches):
            start, end = match.span()
            matched_str = match.group(0)

            # Get the replacement string
            new_str = match.expand(repl) if isinstance(repl, str) else repl(match)
            length_diff = len(new_str) - len(matched_str)

            # 1. Update the actual text
            text = text[:start] + new_str + text[end:]

            # 2. Update the mapping array
            for i in range(len(old_to_new)):
                if old_to_new[i] >= end:
                    # Indices after the replacement shift by the difference in length
                    old_to_new[i] += length_diff
                elif start < old_to_new[i] < end:
                    # If a sloppy entity boundary fell INSIDE the deleted spaces,
                    # snap it safely to the start of the replacement
                    old_to_new[i] = start + min(old_to_new[i] - start, len(new_str))

    # Re-map the entities using the updated index map
    normalized_entities = []
    for ent in raw_entities:
        n_start = old_to_new[ent["start"]]
        n_end = old_to_new[ent["end"]]
        n_text = text[n_start:n_end]

        # Safety: Drop entities that became entirely empty/whitespace after normalization
        if not n_text.strip():
            continue

        normalized_entities.append({
            "id": ent["id"],
            "type": ent["type"],
            "start": n_start,
            "end": n_end,
            "text": n_text
        })

    return text, normalized_entities

In [ ]:
# --- Helper: parse a single .ann file ---
def parse_ann_file(ann_path):
    """Parse a Brat .ann file, handling both format variants."""
    entities = []
    with open(ann_path) as f:
        for line in f:
            line = line.strip()
            if not line or not line.startswith("T"):
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                continue

            annotation_id = parts[0]
            info = parts[1]
            info_parts = info.split()

            if len(info_parts) >= 3:
                # Format: T1\tPERS 6 22\tОлена Петренко
                entity_type = info_parts[0]
                start = int(info_parts[1])
                # Handle semicolons in multi-span: "PERS 6 12;14 22"
                end_str = info_parts[-1].split(";")[-1] if ";" in info_parts[-1] else info_parts[-1]
                end = int(end_str)
                entity_text = parts[2]
            elif len(info_parts) == 1 and len(parts) >= 4:
                # Format: T1\tLOC\t112\t119\tУкраїни
                entity_type = info_parts[0]
                start = int(parts[2])
                end = int(parts[3])
                entity_text = parts[4] if len(parts) > 4 else ""
            else:
                continue

            actual_text = raw_text[start:end]
            if actual_text != entity_text:
                # Option A: Strict - drop the entity
                # continue

                # Option B: Lenient - keep the actual text from the file
                entity_text = actual_text

            entities.append({
                "id": annotation_id,
                "type": entity_type,
                "start": start,
                "end": end,
                "text": entity_text
            })
    return entities

In [ ]:
data_dir = "ner-uk/v2.0/data"
subcorpora = ["bruk", "ng"]

all_examples = []
for subcorpus in subcorpora:
    corpus_dir = os.path.join(data_dir, subcorpus)
    txt_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith(".txt")])

    for txt_file in txt_files:
        name = txt_file.replace(".txt", "")
        txt_path = os.path.join(corpus_dir, txt_file)
        ann_path = os.path.join(corpus_dir, name + ".ann")

        with open(txt_path) as f:
            raw_text = f.read()

        raw_entities = parse_ann_file(ann_path) if os.path.exists(ann_path) else []

        if raw_entities:
            norm_text, norm_entities = normalize_and_map_offsets(raw_text, raw_entities)
        else:
            norm_text, norm_entities = normalize_and_map_offsets(raw_text, [])

        # Determine split
        if name in test_files:
            split = "test"
        elif name in dev_files:
            split = "dev"
        else:
            split = "unknown"

        all_examples.append({
            "name": name,
            "subcorpus": subcorpus,
            "split": split,
            "text": norm_text,
            "entities": norm_entities
        })

print(f"\nTotal loaded: {len(all_examples)} texts")



Total loaded: 560 texts


In [ ]:

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

# Split counts
split_counts = Counter(ex["split"] for ex in all_examples)
print(f"\nBy split: {dict(split_counts)}")

# Subcorpus counts
sub_counts = Counter(ex["subcorpus"] for ex in all_examples)
print(f"By subcorpus: {dict(sub_counts)}")

# Total entities
all_entities = [ent for ex in all_examples for ent in ex["entities"]]
print(f"\nTotal entities: {len(all_entities)}")

# Entity type distribution
type_counts = Counter(ent["type"] for ent in all_entities)
print(f"\nEntity type distribution (sorted by frequency):")
for etype, count in type_counts.most_common():
    print(f"  {etype:8s}: {count:6d}  ({count/len(all_entities)*100:.1f}%)")

# Text length stats
text_lengths = [len(ex["text"]) for ex in all_examples]
entities_per_doc = [len(ex["entities"]) for ex in all_examples]
print(f"\nText length (chars): min={min(text_lengths)}, max={max(text_lengths)}, "
      f"avg={sum(text_lengths)/len(text_lengths):.0f}")
print(f"Entities per doc:    min={min(entities_per_doc)}, max={max(entities_per_doc)}, "
      f"avg={sum(entities_per_doc)/len(entities_per_doc):.1f}")

DATASET OVERVIEW

By split: {'test': 169, 'dev': 391}
By subcorpus: {'bruk': 262, 'ng': 298}

Total entities: 21993

Entity type distribution (sorted by frequency):
  PERS    :   6235  (28.3%)
  ORG     :   5213  (23.7%)
  LOC     :   3000  (13.6%)
  DATE    :   2047  (9.3%)
  JOB     :   1982  (9.0%)
  MON     :    943  (4.3%)
  ART     :    635  (2.9%)
  PERIOD  :    596  (2.7%)
  MISC    :    515  (2.3%)
  QUANT   :    382  (1.7%)
  PCT     :    263  (1.2%)
  DOC     :    142  (0.6%)
  TIME    :     40  (0.2%)

Text length (chars): min=778, max=17075, avg=3351
Entities per doc:    min=0, max=267, avg=39.3


In [ ]:
print("\n" + "=" * 60)
print("EXAMPLE 1 (bruk subcorpus)")
print("=" * 60)
bruk_example = next(ex for ex in all_examples if ex["subcorpus"] == "bruk" and len(ex["entities"]) > 3)
print(f"File: {bruk_example['name']}")
print(f"Split: {bruk_example['split']}")
print(f"Text (first 300 chars):\n{bruk_example['text'][:300]}...\n")
print(f"Entities ({len(bruk_example['entities'])} total):")
for ent in bruk_example["entities"][:10]:
    print(f"  [{ent['type']:6s}] \"{ent['text']}\"  (chars {ent['start']}-{ent['end']})")
if len(bruk_example["entities"]) > 10:
    print(f"  ... and {len(bruk_example['entities']) - 10} more")


EXAMPLE 1 (bruk subcorpus)
File: 0046ebeff13e
Split: test
Text (first 300 chars):
Вивчали хімічний режим підземних вод, які використовуються для напування тварин у західній біогеохімічній зоні України.

За органолептичними показниками та хімічним складом вони відповідають чинним вимогам.

Проте у воді досліджуваних господарств виявили меркурій, що досить небезпечно, а вміст ферум...

Entities (37 total):
  [LOC   ] "України"  (chars 111-118)
  [LOC   ] "Землі"  (chars 395-400)
  [PCT   ] "75%"  (chars 480-483)
  [JOB   ] "академіком"  (chars 1019-1029)
  [PERS  ] "В. І. Вернадським"  (chars 1030-1047)
  [PERS  ] "Д. І. Менделєєва"  (chars 1264-1280)
  [LOC   ] "України"  (chars 1596-1603)
  [LOC   ] "України"  (chars 1687-1694)
  [LOC   ] "Волинській"  (chars 1697-1707)
  [LOC   ] "Київській"  (chars 1709-1718)
  ... and 27 more


In [ ]:
print("\n" + "=" * 60)
print("EXAMPLE 2 (ng subcorpus)")
print("=" * 60)
ng_example = next(ex for ex in all_examples if ex["subcorpus"] == "ng" and len(ex["entities"]) > 5)
print(f"File: {ng_example['name']}")
print(f"Split: {ng_example['split']}")
print(f"Text (first 300 chars):\n{ng_example['text'][:300]}...\n")
print(f"Entities ({len(ng_example['entities'])} total):")
for ent in ng_example["entities"][:10]:
    print(f"  [{ent['type']:6s}] \"{ent['text']}\"  (chars {ent['start']}-{ent['end']})")
if len(ng_example["entities"]) > 10:
    print(f"  ... and {len(ng_example['entities']) - 10} more")


EXAMPLE 2 (ng subcorpus)
File: 003d28360166
Split: dev
Text (first 300 chars):
ПАТ «Укртрансгаз» 29 квітня за результатами тендеру уклало угоду з ТОВ «Укргеоекологія» на проведення газогеохімічних та гідрогеологічних досліджень на підземних газосховищах за 7,08 млн грн.
Про це повідомляється у «Віснику державних закупівель».
Дослідження мають провести на 12 газосховищах у 5 уп...

Entities (50 total):
  [ORG   ] "ПАТ «Укртрансгаз»"  (chars 0-17)
  [DATE  ] "29 квітня"  (chars 18-27)
  [ORG   ] "ТОВ «Укргеоекологія»"  (chars 67-87)
  [MON   ] "7,08 млн грн"  (chars 178-190)
  [MISC  ] "Віснику державних закупівель"  (chars 217-245)
  [ORG   ] "Прикарпаттрансгаз"  (chars 342-359)
  [LOC   ] "Богородчанське ПСГ"  (chars 362-380)
  [ORG   ] "Львівтрансгаз"  (chars 391-404)
  [LOC   ] "Дашавське"  (chars 407-416)
  [LOC   ] "Опарське"  (chars 418-426)
  ... and 40 more


# Transform brat to prompt-response

Викинемо деякі клас, бо вони мало представлені у датасеті.

Keep (8 types):

PERS (6,235) — largest class, core NER entity, must keep
ORG (5,213) — second largest, core NER entity
LOC (3,000) — core NER entity
DATE (2,047) — plenty of examples, well-defined type
JOB (1,982) — plenty of examples, interesting non-standard entity type
MON (943) — enough examples, well-defined (number + currency)
ART (635) — borderline but sufficient, adds variety
PERIOD (596) — similar to ART, enough data

Drop (5 types):

MISC (515) — despite having decent count, the paper itself says this type is too broadly defined. Even the roberta-large baseline only achieved 0.35 F1 on it. If a dedicated fine-tuned NER model can't handle it, a generative LLM won't either. It would just add noise to your metrics.
QUANT (382) — getting small, and it overlaps conceptually with MON and PCT (all are "number + unit" patterns)
PCT (263) — too few, and it's a very simple pattern (number + %). Not very interesting for NER evaluation.
DOC (142) — way too few. Even in the paper, DOC got only 0.44 F1. With ~40 examples in the test set, your metrics would be unreliable.
TIME (40) — only 10 in the test set. Statistically meaningless. One correct or incorrect prediction would swing F1 wildly.

In [ ]:
# === Brat to entity-list format conversion ===

import re

# Entity types we keep
KEEP_ENTITIES = {
    "PERS", "ORG", "LOC", "DATE", "JOB", "MON", "ART", "PERIOD",
    "MISC", "QUANT", "PCT", "DOC", "TIME"
}

In [ ]:
def brat_to_inline_tagged(text, entities):
    """Convert Brat annotations to inline-tagged text: [TYPE: entity]."""
    filtered = [e for e in entities if e["type"] in KEEP_ENTITIES]
    filtered.sort(key=lambda e: (e["start"], -(e["end"] - e["start"])))

    clean = []
    last_end = -1
    for ent in filtered:
        # Safety check: do not include entities that are empty or pure whitespace
        if ent["start"] >= last_end and ent["text"].strip():
            clean.append(ent)
            last_end = ent["end"]

    result = []
    prev_end = 0
    for ent in clean:
        result.append(text[prev_end:ent["start"]])
        entity_text = text[ent["start"]:ent["end"]]

        if not entity_text.strip():
            continue # Prevent creating empty tags like [PERS: ]

        result.append(f"[{ent['type']}: {entity_text}]")
        prev_end = ent["end"]
    result.append(text[prev_end:])

    # Final cleanup: just in case tag insertion created double spaces
    final_text = "".join(result)
    final_text = re.sub(r' {2,}', ' ', final_text)

    return final_text, len(clean)

In [ ]:
# === Convert all examples ===
train_pairs = []
test_pairs = []
total_dropped_nested = 0

for ex in all_examples:
    all_kept = [e for e in ex["entities"] if e["type"] in KEEP_ENTITIES]
    tagged_text, actual_count = brat_to_inline_tagged(ex["text"], ex["entities"])
    dropped = len(all_kept) - actual_count
    total_dropped_nested += dropped

    pair = {
        "name": ex["name"],
        "subcorpus": ex["subcorpus"],
        "input_text": ex["text"],
        "tagged_text": tagged_text,
        "num_entities": actual_count,
        "num_dropped_nested": dropped
    }

    if ex["split"] == "test":
        test_pairs.append(pair)
    elif ex["split"] == "dev":
        train_pairs.append(pair)

print(f"Train pairs: {len(train_pairs)}")
print(f"Test pairs:  {len(test_pairs)}")
print(f"Total entities: {sum(p['num_entities'] for p in train_pairs + test_pairs)}")
print(f"Total nested entities dropped: {total_dropped_nested}")

Train pairs: 391
Test pairs:  169
Total entities: 21556
Total nested entities dropped: 437


On dropping nested entities: this is totally fine and standard practice. The paper itself says that when converting to IOB format, they discard nested annotations. Most NER systems don't handle nesting. The typical example is: [PERIOD: [DATE: 2011]-[DATE: 2012] рр.] — the PERIOD contains two DATEs inside. We keep PERIOD and drop the inner DATEs. You lose ~341 entities out of ~20,600 — that's about 1.6%. Definitely worth mentioning in your report, but not a problem.

In [ ]:
# === Verify: show a few examples ===
print("\n" + "=" * 60)
print("EXAMPLE 1")
print("=" * 60)
ex = next(p for p in train_pairs if p["num_entities"] > 3)
print(f"File: {ex['name']} ({ex['subcorpus']})")
print(f"Entities: {ex['num_entities']}")
print(f"\nORIGINAL (first 300 chars):")
print(ex["input_text"])
print(f"\nTAGGED (first 400 chars):")
print(ex["tagged_text"])
if len(ex["tagged_text"]) > 400:
    print(f"  ... ({len(ex['tagged_text'])} chars total)")

print("\n" + "=" * 60)
print("EXAMPLE 2")
print("=" * 60)
ex2 = next(p for p in train_pairs if p["subcorpus"] == "ng" and p["num_entities"] > 5)
print(f"File: {ex2['name']} ({ex2['subcorpus']})")
print(f"Entities: {ex2['num_entities']}")
print(f"\nORIGINAL (first 300 chars):")
print(ex2["input_text"])
print(f"\nTAGGED (first 400 chars):")
print(ex2["tagged_text"])
if len(ex2["tagged_text"]) > 400:
    print(f"  ... ({len(ex2['tagged_text'])} chars total)")


EXAMPLE 1
File: 0124bf2fdf35 (bruk)
Entities: 33

ORIGINAL (first 300 chars):
Дорогою зупинялися в селах, старцювали.
Устимко приспівував дідам.
Амфібрахій шукав підробіток.

То молитви продавав свої спудейські, то щось комусь косив чи рубав, а бувало, хтось приходив до діда і просив якогось зілля на лік і тоді дід, пояснивши Амфібрахію, яку саме траву треба принести, посилав його на пошуки.

Дуже часто дід досить точно вказував, куди саме йти і де рвати чи копати, що при тім примовити, що покласти на землю.

Часом Амфібрахій, керуючись настановами старого кобзаря, знаходив траву дуже швидко, але чекав аж до заходу сонця, або доки на небі згасне остання зірка, або до іншої якої пори, - тільки тоді можна було зірвати її так, щоб вона мала силу прогнати духа хвороби.

Чи вірив у все це сам Амфібрахій?

Мабуть, ні.

Але так треба було зробити.

Саме так, а не інакше.

Тож і робив, як веліли.

Якісь звичні трави, такі як материнка, ромашка, липовий цвіт чи полин, знали всі, тож Амфібрахій

In [ ]:
def parse_inline_tagged_output(response):
    type_aliases = {
        "PERS": "PERS", "PERSON": "PERS", "PER": "PERS",
        "ORG": "ORG", "ORGANISATION": "ORG", "ORGANIZATION": "ORG",
        "LOC": "LOC", "LOCATION": "LOC",
        "DATE": "DATE",
        "JOB": "JOB",
        "MON": "MON", "MONEY": "MON",
        "ART": "ART", "ARTIFACT": "ART",
        "PERIOD": "PERIOD",
        "MISC": "MISC", "MISCELLANEOUS": "MISC",
        "QUANT": "QUANT", "QUANTITY": "QUANT",
        "PCT": "PCT", "PERCENT": "PCT", "PERCENTAGE": "PCT",
        "DOC": "DOC", "DOCUMENT": "DOC",
        "TIME": "TIME",
    }

    entities = []
    for match in re.finditer(r'\[([A-Za-z]+):\s*(.+?)\]', response, re.DOTALL):
        raw_type = match.group(1).strip().upper()
        ent_text = re.sub(r'\s+', ' ', match.group(2).strip())
        mapped_type = type_aliases.get(raw_type)
        if mapped_type and ent_text:
            entities.append((mapped_type, ent_text))
    return entities

In [ ]:
# === Round-trip verification ===
print("\n" + "=" * 60)
print("ROUND-TRIP VERIFICATION")
print("=" * 60)

mismatches = 0
for p in train_pairs + test_pairs:
    extracted = parse_inline_tagged_output(p["tagged_text"])
    if len(extracted) != p["num_entities"]:
        print(f"MISMATCH: {p['name']} — expected {p['num_entities']}, extracted {len(extracted)}")
        # Show first few extracted for debugging
        for e in extracted[:5]:
            print(f"    {e}")
        mismatches += 1

if mismatches == 0:
    print(f"All {len(train_pairs) + len(test_pairs)} examples pass round-trip verification!")
else:
    print(f"{mismatches} mismatches found — needs investigation")


ROUND-TRIP VERIFICATION
All 560 examples pass round-trip verification!


# Prompt template

In [ ]:
# SYSTEM_PROMPT = """
# Ти — інструмент точної розмітки тексту, а не редактор. Перепиши текст, позначивши іменовані сутності у форматі [ТИП: сутність].

# Типи:
# PERS — імена людей, літературних персонажів та істот
# ORG — організації, компанії, установи, партії, проєкти, конференції
# LOC — географічні назви: райони, міста, країни, ріки, озера, моря, гори
# DATE — повна або часткова календарна дата (століття, рік, місяць, день)
# TIME — текстова або числова мітка часу (година, хвилина)
# PERIOD — часовий проміжок, може складатися з двох дат або одним словом
# JOB — посади, професії та звання: поет, директор, суддя, академік, лікар
# MON — грошова сума з валютою
# PCT — відсоткове значення зі знаком % або словом "відсотків"
# QUANT — кількість з одиницею виміру (вага, відстань, розмір)
# ART — назви створених людиною продуктів: книги, пісні, фільми, автомобілі, бренди
# DOC — унікальні назви документів: контракти, закони, накази, договори
# MISC — інші іменовані сутності: свята, веб-сайти, битви, війни, спортивні події(Великдень, Вісник, Закон)

# Правила:
# - Виведи ВЕСЬ текст, додаючи позначки [ТИП: сутність] навколо знайдених сутностей.
# - ЗАБОРОНЕНО редагувати текст: не виправляй орфографію, не розшифровуй абревіатури, не перефразуй. Текст поза сутностями має залишитися АБСОЛЮТНО БЕЗ ЗМІН.
# - Копіюй сутності ТОЧНО як в оригіналі. Зберігай початковий відмінок (не приводь до називного) та оригінальну форму слова.
# - НЕ захоплюй у теги прилеглі розділові знаки (крапки наприкінці речень, коми, лапки), якщо вони не є невід'ємною частиною самої сутності.
# - Використовуй ТІЛЬКИ скорочення: PERS, ORG, LOC, DATE, JOB, MON, ART, PERIOD, PCT, QUANT, DOC, MISC, TIME.
# """

In [ ]:
SYSTEM_PROMPT = """
Ти — інструмент прецизійної розмітки тексту (Named Entity Recognition), а не редактор чи автор. Твоє завдання — ідентифікувати іменовані сутності в українському тексті та позначити їх у форматі [ТИП: сутність].\n\n### СЛОВНИК ТИПІВ СУТНОСТЕЙ:\n- **PERS** — імена людей, прізвища, літературні персонажі, міфічні істоти, божества та сакральні особи (наприклад, \"Господа\"). \n    *   *Увага:* Не позначай як PERS загальні абревіатури (наприклад, \"ПІБ\" не є сутністю PERS).\n- **ORG** — конкретні організації, компанії, установи, партії, проєкти, конференції, релігійні інституції (наприклад, \"Церква\", \"КВП\", \"ТОВ «Спектр»\").\n    *   *Увага:* Не позначай як ORG прикметники національності.\n- **LOC** — реальні географічні назви: райони, міста, країни, ріки, озера, моря, гори, повні адреси.\n    *   **Розділення:** Якщо в тексті вказано місто та область (наприклад, \"Буча Київської області\"), позначай їх як дві окремі сутності: [LOC: Буча] [LOC: Київської області].\n    *   **Виняток:** Якщо назва географічного об'єкта є вигаданою, поетичною або метафоричною (наприклад, \"Медове місто\"), позначай її як **MISC**.\n- **DATE** — повна або часткова календарна дата (століття, рік, місяць, день).\n- **TIME** — текстова або числова мітка часу (година, хвилина).\n- **PERIOD** — часовий проміжок, тривалість або відносна дата (наприклад, \"п'ять років\", \"2000-х\", \"тиждень\", \"останні п'ять років\").\n- **JOB** — посади, професії, звання, соціальні та релігійні ролі (наприклад, \"поет\", \"директор\", \"керівник\", \"засновник\", \"пастирів\", \"депутатів\", \"фінансовим директором\").\n- **MON** — грошова сума з валютою (наприклад, \"17,81 млн грн\", \"5 тис грн\").\n- **PCT** — відсоткове значення зі знаком % або словом \"відсотків\" / \"піввідсотка\".\n- **QUANT** — кількість з одиницею виміру (вага, відстань, площа, розмір, погонний метр). Включай число (навіть словами) та одиницю в один тег (наприклад, \"[QUANT: один погонний метр]\", \"[QUANT: 2 га]\").\n- **ART** — назви творів, продуктів, брендів, періодичних видань, газет та журналів (наприклад, \"Віснику державних закупівель\", \"Євангелія\", \"Ізопрофлекс\").\n- **DOC** — унікальні назви конкретних документів, юридичних актів, угод, рішень, кримінальних проваджень.\n    *   **Межі:** Захоплюй всю назву документа разом із номером та датою (наприклад, \"[DOC: кримінальне провадження № 123 від 01.01.2020]\").\n    *   **Пріоритет:** DOC $\\rightarrow$ (ORG, DATE). Якщо дата або організація є частиною назви документа, вони поглинаються тегом DOC.\n- **MISC** — інші іменовані сутності: свята, веб-сайти, битви, війни, спортивні події, релігійні поняття як назви (\"Великдень\", \"Літургії\"), вигадані топоніми, а також назви конкретних об'єктів будівництва або проєктів реконструкції (наприклад, \"[MISC: Реконструкція транспортної розв'язки на Поштовій площі м. Києва]\").\n\n### СУВОРИЙ ПРОТОКОЛ ВИКОНАННЯ:\n1. **Абсолютна цілісність тексту (Zero Editing):** Виведи ВЕСЬ наданий текст. ЗАБОРОНЕНО редагувати, виправляти орфографію, розшифровувати абревіатури, змінювати пунктуацію або перефразувати. \n2. **Заборона доповнень та галюцинацій (No Hallucinations):** \n    - КАТЕГОРИЧНО ЗАБОРОНЕНО додавати слова, речення або цілі абзаци, яких немає у вхідному тексті.\n    - Якщо текст обривається на середині слова або речення — залиш його обірваним.\n    - Не дописуй назви організацій або документів, використовуючи свої знання; використовуй ТІЛЬКИ ті слова, що є в тексті.\n3. **Точність копіювання:** Копіюй сутності ТОЧНО як в оригіналі. Зберігай початковий відмінок, регістр та форму слова.\n4. **Межі сутностей:** \n    - Не захоплюй у теги розділові знаки (коми, крапки), якщо вони не є частиною офіційної назви.\n    - Захоплюй усі визначники, що є частиною сутності. \n    - **Повні назви:** Завжди захоплюй повну назву організації (наприклад, не просто \"[ORG: Львівської міськради]\", а \"[ORG: Управління капітального будівництва Львівської міськради]\").\n    - **Заборона на загальні іменники:** Не позначай як сутності загальні назви без власної назви (наприклад, слово \"місті\" або \"ріка\" само по собі не є LOC).\n5. **Пріоритетність:** DOC $\\rightarrow$ (ORG, DATE).\n\n### ПРИКЛАД ПРАВИЛЬНОЇ РОЗМІТКИ:\n*Вхід:* \"Згідно з рішенням міськради від 10 січня 2020 року, ділянку площею 2 га передали ТОВ «Спектр». Про це писали у «Віснику закупівель».\"\n*Вихід:* \"Згідно з [DOC: рішенням міськради від 10 січня 2020 року], ділянку площею [QUANT: 2 га] передали [ORG: ТОВ «Спектр»]. Про це писали у [ART: «Віснику закупівель»].\"
"""

In [ ]:
def format_ner_prompt(text, system_prompt=None):
    prompt = system_prompt if system_prompt is not None else SYSTEM_PROMPT
    return f"{prompt}\n\nТекст:\n{text}"


def create_chat_messages(text, system_prompt=None):
    return [{"role": "user", "content": format_ner_prompt(text, system_prompt=system_prompt)}]


def create_training_messages(text, tagged_text):
    return [
        {"role": "user", "content": format_ner_prompt(text)},
        {"role": "assistant", "content": tagged_text}
    ]

In [ ]:
print("=" * 60)
print("PROMPT EXAMPLE (short)")
print("=" * 60)

test_text = "ПАТ «Укртрансгаз» 29 квітня уклало угоду з ТОВ «Укргеоекологія» за 7,08 млн грн."
messages = create_chat_messages(test_text)
print(messages[0]["content"])

print("\n" + "=" * 60)
print("EXPECTED RESPONSE")
print("=" * 60)

test_response = "[ORG: ПАТ «Укртрансгаз»] [DATE: 29 квітня] уклало угоду з [ORG: ТОВ «Укргеоекологія»] за [MON: 7,08 млн грн]."
print(test_response)

PROMPT EXAMPLE (short)

Ти — інструмент прецизійної розмітки тексту (Named Entity Recognition), а не редактор чи автор. Твоє завдання — ідентифікувати іменовані сутності в українському тексті та позначити їх у форматі [ТИП: сутність].

### СЛОВНИК ТИПІВ СУТНОСТЕЙ:
- **PERS** — імена людей, прізвища, літературні персонажі, міфічні істоти, божества та сакральні особи (наприклад, "Господа"). 
    *   *Увага:* Не позначай як PERS загальні абревіатури (наприклад, "ПІБ" не є сутністю PERS).
- **ORG** — конкретні організації, компанії, установи, партії, проєкти, конференції, релігійні інституції (наприклад, "Церква", "КВП", "ТОВ «Спектр»").
    *   *Увага:* Не позначай як ORG прикметники національності.
- **LOC** — реальні географічні назви: райони, міста, країни, ріки, озера, моря, гори, повні адреси.
    *   **Розділення:** Якщо в тексті вказано місто та область (наприклад, "Буча Київської області"), позначай їх як дві окремі сутності: [LOC: Буча] [LOC: Київської області].
    *   **Виняток

In [ ]:
print("\n" + "=" * 60)
print("TOKEN LENGTH ANALYSIS")
print("=" * 60)

# 1. Count the raw tokens of the SYSTEM_PROMPT string alone
system_prompt_tokens = len(tokenizer(SYSTEM_PROMPT, add_special_tokens=False).input_ids)
print(f"System Prompt (Raw Tokens): {system_prompt_tokens}")

# 2. Count the TRUE OVERHEAD (System Prompt + Chat Template formatting tags)
# We pass an empty string "" to see exactly how many tokens the template itself takes up
dummy_messages = create_chat_messages("")
dummy_prompt_string = tokenizer.apply_chat_template(dummy_messages, tokenize=False, add_generation_prompt=True)
true_overhead_tokens = len(tokenizer(dummy_prompt_string, add_special_tokens=False).input_ids)
print(f"Total Prompt Overhead (System + Gemma Tags): {true_overhead_tokens}")

# 3. Analyze a real training example
train_messages = create_training_messages(test_text, test_response)
full_train_string = tokenizer.apply_chat_template(train_messages, tokenize=False)
total_sequence_tokens = len(tokenizer(full_train_string, add_special_tokens=False).input_ids)

print(f"\nExample Total Sequence Tokens: {total_sequence_tokens}")


TOKEN LENGTH ANALYSIS
System Prompt (Raw Tokens): 1571
Total Prompt Overhead (System + Gemma Tags): 1581

Example Total Sequence Tokens: 1681


In [ ]:
print("\n" + "=" * 60)
print("TOKEN LENGTH ANALYSIS")
print("=" * 60)

token_lengths = []
for p in train_pairs:
    msgs = create_training_messages(p["input_text"], p["tagged_text"])
    full_prompt = tokenizer.apply_chat_template(msgs, tokenize=False)
    tokens = tokenizer(full_prompt, return_tensors="pt")
    token_lengths.append(tokens.input_ids.shape[1])

print(f"Training examples: {len(token_lengths)}")
print(f"Token length — min: {min(token_lengths)}, max: {max(token_lengths)}, avg: {sum(token_lengths)/len(token_lengths):.0f}")
print(f"Median: {sorted(token_lengths)[len(token_lengths)//2]}")


TOKEN LENGTH ANALYSIS
Training examples: 391
Token length — min: 2239, max: 12443, avg: 4027
Median: 3208


In [ ]:
# How many exceed common context windows
for limit in [1024, 2048, 4096, 8192, 10240, 12288]:
    over = sum(1 for t in token_lengths if t > limit)
    print(f"Examples over {limit} tokens: {over} ({over/len(token_lengths)*100:.1f}%)")

Examples over 1024 tokens: 391 (100.0%)
Examples over 2048 tokens: 391 (100.0%)
Examples over 4096 tokens: 115 (29.4%)
Examples over 8192 tokens: 17 (4.3%)
Examples over 10240 tokens: 7 (1.8%)
Examples over 12288 tokens: 1 (0.3%)


# Text chunking

In [ ]:
def split_text_into_chunks(text, entities, tokenizer,
                           max_total_tokens=1500,
                           prompt_overhead_tokens=416,
                           overlap_fraction=0.2,     # 20% перекриття
                           is_training=True):        # Прапорець для перекриття
    """
    Split a long text into chunks on paragraph/sentence boundaries.
    Applies sliding window overlap ONLY during training.
    """
    text_token_budget = int((max_total_tokens - prompt_overhead_tokens) / 2.2)
    # Бюджет перекриття діє лише для тренувальних даних
    overlap_budget = int(text_token_budget * overlap_fraction) if is_training else 0

    # --- Step 1: Split text into paragraphs/sentences (ЗАЛИШАЄТЬСЯ БЕЗ ЗМІН) ---
    paragraphs = []
    current_pos = 0
    for part in text.split("\n\n"):
        start = text.index(part, current_pos)
        end = start + len(part)
        paragraphs.append({"text": part, "start": start, "end": end})
        current_pos = end

    refined = []
    for para in paragraphs:
        para_tokens = len(tokenizer(para["text"], add_special_tokens=False).input_ids)
        if para_tokens <= text_token_budget:
            refined.append(para)
        else:
            import re
            sentences = re.split(r'(?<=[\.\!\?])\s+', para["text"])
            sent_pos = para["start"]
            for sent in sentences:
                s_start = text.index(sent, sent_pos)
                s_end = s_start + len(sent)
                refined.append({"text": sent, "start": s_start, "end": s_end})
                sent_pos = s_end

    # --- Step 2: Group segments into chunks (SLIDING WINDOW) ---
    chunks = []
    start_idx = 0

    while start_idx < len(refined):
        current_segments = []
        current_tokens = 0
        end_idx = start_idx

        # Будуємо чанк вперед, поки є місце
        while end_idx < len(refined):
            seg = refined[end_idx]
            seg_tokens = len(tokenizer(seg["text"], add_special_tokens=False).input_ids)
            if current_tokens + seg_tokens > text_token_budget and current_segments:
                break
            current_segments.append(seg)
            current_tokens += seg_tokens
            end_idx += 1

        # Зберігаємо чанк
        chunk_start = current_segments[0]["start"]
        chunk_end = current_segments[-1]["end"]
        chunks.append((text[chunk_start:chunk_end], chunk_start, chunk_end))

        if end_idx == len(refined):
            break  # Дійшли до кінця документа

        # Визначаємо, звідки почнеться НАСТУПНИЙ чанк
        if is_training and overlap_budget > 0:
            overlap_tokens = 0
            next_start = end_idx - 1

            # Відступаємо назад, щоб захопити частину попереднього тексту
            while next_start > start_idx:
                past_tokens = len(tokenizer(refined[next_start]["text"], add_special_tokens=False).input_ids)
                if overlap_tokens + past_tokens > overlap_budget:
                    next_start += 1 # Крок вперед, щоб не перевищити бюджет перекриття
                    break
                overlap_tokens += past_tokens
                next_start -= 1

            # Гарантуємо, що ми завжди рухаємося вперед (уникаємо нескінченного циклу)
            if next_start <= start_idx:
                next_start = start_idx + 1
            start_idx = next_start
        else:
            # Для тестування: наступний чанк починається рівно там, де закінчився попередній
            start_idx = end_idx

    # --- Step 3: Assign entities to chunks (ЗАЛИШАЄТЬСЯ БЕЗ ЗМІН) ---
    result = []
    for chunk_text, chunk_start, chunk_end in chunks:
        chunk_entities = []
        for ent in entities:
            if ent["start"] >= chunk_start and ent["end"] <= chunk_end:
                chunk_entities.append({
                    "id": ent["id"],
                    "type": ent["type"],
                    "start": ent["start"] - chunk_start,
                    "end": ent["end"] - chunk_start,
                    "text": ent["text"]
                })
        result.append((chunk_text, chunk_entities))

    return result

In [ ]:
def build_chunked_dataset(examples, tokenizer, max_total_tokens=1500):
    """Process all examples: chunk long texts, keep short ones as-is."""
    chunked = []
    prompt_overhead = 1581

    for ex in examples:
        kept_entities = [e for e in ex["entities"] if e["type"] in KEEP_ENTITIES]

        # Перевіряємо, чи це тренувальний набір
        is_train = (ex["split"] == "dev")

        raw_input_tokens = len(tokenizer(ex["text"], add_special_tokens=False).input_ids)
        estimated_output_tokens = int(raw_input_tokens * 1.2)
        estimated_full_sequence = prompt_overhead + raw_input_tokens + estimated_output_tokens

        if estimated_full_sequence <= max_total_tokens:
            tagged_text, num_ents = brat_to_inline_tagged(ex["text"], kept_entities)

            chunked.append({
                "name": f"{ex['name']}_chunk_0",
                "original_name": ex["name"],
                "subcorpus": ex["subcorpus"],
                "split": ex["split"],
                "input_text": ex["text"],
                "tagged_text": tagged_text,
                "num_entities": num_ents
            })
        else:
            # Передаємо прапорець is_training у функцію нарізки
            text_chunks = split_text_into_chunks(
                ex["text"], kept_entities, tokenizer, max_total_tokens, prompt_overhead,
                is_training=is_train, overlap_fraction=0.25
            )

            for i, (chunk_text, chunk_entities) in enumerate(text_chunks):
                tagged_text, num_ents = brat_to_inline_tagged(chunk_text, chunk_entities)

                chunked.append({
                    "name": f"{ex['name']}_chunk_{i}",
                    "original_name": ex["name"],
                    "subcorpus": ex["subcorpus"],
                    "split": ex["split"],
                    "input_text": chunk_text,
                    "tagged_text": tagged_text,
                    "num_entities": num_ents
                })

    return chunked

In [ ]:
print("Building chunked datasets...")
print("This may take a minute due to tokenization...\n")

all_chunked = build_chunked_dataset(all_examples, tokenizer, max_total_tokens=2400)

train_chunks = [c for c in all_chunked if c["split"] == "dev"]
test_chunks = [c for c in all_chunked if c["split"] == "test"]

print(f"Original documents:  {len(all_examples)}")
print(f"After chunking:      {len(all_chunked)} total chunks")
print(f"  Train chunks:      {len(train_chunks)}")
print(f"  Test chunks:       {len(test_chunks)}")

Building chunked datasets...
This may take a minute due to tokenization...

Original documents:  560
After chunking:      2320 total chunks
  Train chunks:      1667
  Test chunks:       653


In [ ]:
print("\nTOKEN LENGTH ANALYSIS (after chunking)")

chunk_token_lengths = []
for c in all_chunked:
    msgs = create_training_messages(c["input_text"], c["tagged_text"])
    full = tokenizer.apply_chat_template(msgs, tokenize=False)
    tokens = len(tokenizer(full, add_special_tokens=False).input_ids)
    chunk_token_lengths.append(tokens)

print(f"Token length — min: {min(chunk_token_lengths)}, max: {max(chunk_token_lengths)}, avg: {sum(chunk_token_lengths)/len(chunk_token_lengths):.0f}")
print(f"Median: {sorted(chunk_token_lengths)[len(chunk_token_lengths)//2]}")

for limit in [768, 1024, 1536, 1792, 2048, 2560, 4096, 8192]:
    over = sum(1 for t in chunk_token_lengths if t > limit)
    print(f"Over {limit} tokens: {over} ({over/len(chunk_token_lengths)*100:.1f}%)")


TOKEN LENGTH ANALYSIS (after chunking)
Token length — min: 1608, max: 2564, avg: 2263
Median: 2325
Over 768 tokens: 2320 (100.0%)
Over 1024 tokens: 2320 (100.0%)
Over 1536 tokens: 2320 (100.0%)
Over 1792 tokens: 2263 (97.5%)
Over 2048 tokens: 2026 (87.3%)
Over 2560 tokens: 1 (0.0%)
Over 4096 tokens: 0 (0.0%)
Over 8192 tokens: 0 (0.0%)


In [ ]:
print("ENTITY STATS (after chunking)")

total_ents = sum(c["num_entities"] for c in all_chunked)
train_ents = sum(c["num_entities"] for c in train_chunks)
test_ents = sum(c["num_entities"] for c in test_chunks)
empty_chunks = sum(1 for c in all_chunked if c["num_entities"] == 0)

print(f"Total entities:   {total_ents}")
print(f"  Train entities: {train_ents}")
print(f"  Test entities:  {test_ents}")
print(f"Empty chunks (no entities): {empty_chunks}")

ENTITY STATS (after chunking)
Total entities:   24094
  Train entities: 17288
  Test entities:  6806
Empty chunks (no entities): 146


In [ ]:
print("EXAMPLE: CHUNKED DOCUMENT")

# Find a document that was split into multiple chunks
from collections import Counter
orig_counts = Counter(c["original_name"] for c in all_chunked)
multi_chunk_doc = next(name for name, count in orig_counts.most_common() if count > 2)
doc_chunks = [c for c in all_chunked if c["original_name"] == multi_chunk_doc]

print(f"Document '{multi_chunk_doc}' was split into {len(doc_chunks)} chunks:\n")
for c in doc_chunks[:5]:  # show first 10 chunks only
    print(f"  {c['name']}: {len(c['input_text'])} chars, {c['num_entities']} entities")
    print(f"  ORIGINAL: {(c['input_text'])}")
    print(f"  TAGGED:   {(c['tagged_text'])}")
    print()
if len(doc_chunks) > 5:
    print(f"  ... and {len(doc_chunks) - 5} more chunks")

EXAMPLE: CHUNKED DOCUMENT
Document '8051a5966e78' was split into 20 chunks:

  8051a5966e78_chunk_0: 1104 chars, 3 entities
  ORIGINAL: ЗАКОН УКРАЇНИ Про добровільне об’єднання територіальних громад

Розділ I

ЗАГАЛЬНІ ПОЛОЖЕННЯ

Стаття 1. Відносини, що регулюються цим Законом

1.

Цей Закон регулює відносини, що виникають у процесі добровільного об’єднання територіальних громад сіл, селищ, міст.

Стаття 2. Принципи добровільного об’єднання територіальних громад

1.

Добровільне об’єднання територіальних громад сіл, селищ, міст здійснюється з дотриманням таких принципів:

1) конституційності та законності;

2) добровільності;

3) економічної ефективності;

4) державної підтримки;

5) повсюдності місцевого самоврядування;

6) прозорості та відкритості;

7) відповідальності.

Стаття 3. Суб’єкти добровільного об’єднання територіальних громад

1.

Суб’єктами добровільного об’єднання територіальних громад є суміжні територіальні громади сіл, селищ, міст.

2.

Об’єднана територіальна громада

# Output parsing

In [ ]:
import unicodedata
import re

def normalize_evaluation_text(text):
    """
    Normalizes entity text for lenient evaluation comparison.
    Applies NFC normalization, mixed-alphabet correction, and aggressive punctuation stripping.
    """
    if not text:
        return ""

    # 1. Unicode NFC normalization (ensures composed characters)
    text = unicodedata.normalize('NFC', text)

    # 2. Latin-to-Cyrillic lookalike replacement (Context-Aware)
    latin_to_cyrillic = str.maketrans(
        "aAeEoOpPcCyYxXiITHKBM",
        "аАеЕоОрРсСуУхХіІТНКВМ"
    )

    def fix_mixed_word(match):
        word = match.group(0)
        # Only translate if the word contains at least one Cyrillic character
        if re.search(r'[А-Яа-яЄєІіЇїҐґ]', word):
            return word.translate(latin_to_cyrillic)
        return word

    text = re.sub(r'\w+', fix_mixed_word, text)

    # 3. Spacing and dash normalization
    text = re.sub(r' {2,}', ' ', text)                           # Multiple spaces to single space
    text = re.sub(r'[–—]', '-', text)                            # Standardize dashes
    text = re.sub(r'(?<=\d) +(-)+ +(?=\d)', r'\1', text)         # Spaces around dashes between digits
    text = re.sub(r'(?<=\d) +(?=%)', '', text)                   # Space before percentage sign

    text = re.sub(r'[«»"\'„“]', '', text)

    # 4. THE FIX: Aggressive Trailing/Leading Punctuation Stripping
    # Це вирішує проблему "1000 грн." проти "1000 грн" і випадкових ком.
    text = text.strip()
    text = text.strip('.,:;!?\"\'«»()[]{}\\/')

    # 5. Final space strip (in case punctuation removal left hanging spaces)
    return text.strip()


In [ ]:
def parse_inline_tagged_output(response):
    """Parse [TYPE: entity] markers from inline-tagged text."""
    type_aliases = {
        "PERS": "PERS", "PERSON": "PERS", "PER": "PERS",
        "ORG": "ORG", "ORGANISATION": "ORG", "ORGANIZATION": "ORG",
        "LOC": "LOC", "LOCATION": "LOC",
        "DATE": "DATE",
        "JOB": "JOB",
        "MON": "MON", "MONEY": "MON",
        "ART": "ART", "ARTIFACT": "ART",
        "PERIOD": "PERIOD",
        "MISC": "MISC", "MISCELLANEOUS": "MISC",
        "QUANT": "QUANT", "QUANTITY": "QUANT",
        "PCT": "PCT", "PERCENT": "PCT", "PERCENTAGE": "PCT",
        "DOC": "DOC", "DOCUMENT": "DOC",
        "TIME": "TIME",
    }

    entities = []

    # THE FIX: Стійкий, нежадібний (lazy) регулярний вираз.
    # Він ігнорує зайві пробіли біля дужок та двокрапки: [ ORG : Text ]
    pattern = r'\[\s*([A-Za-z]+)\s*:\s*(.*?)\s*\]'

    for match in re.finditer(pattern, response, flags=re.DOTALL):
        raw_type = match.group(1).strip().upper()
        raw_text = match.group(2)

        # Відкидаємо порожні теги, якщо модель згенерувала [ORG: ]
        if not raw_text.strip():
            continue

        # Пропускаємо текст через оновлений нормалізатор
        ent_text = normalize_evaluation_text(raw_text)
        mapped_type = type_aliases.get(raw_type)

        if mapped_type and ent_text:
            entities.append((mapped_type, ent_text))

    return entities

# Eval metrics

In [ ]:
from collections import Counter
import re

def compute_ner_metrics(chunk_golds, chunk_preds, mode="lenient"):
    if chunk_golds and not isinstance(chunk_golds[0], list):
        chunk_golds = [chunk_golds]
    if chunk_preds and not isinstance(chunk_preds[0], list):
        chunk_preds = [chunk_preds]

    tp_total = 0
    fp_total = 0
    fn_total = 0
    type_stats = {}

    for gold_entities, pred_entities in zip(chunk_golds, chunk_preds):
        if mode == "lenient":
            gold_normalized = [(t, normalize_evaluation_text(txt)) for t, txt in gold_entities]
            pred_normalized = [(t, normalize_evaluation_text(txt)) for t, txt in pred_entities]
        else:
            gold_normalized = [(t, re.sub(r'\s+', ' ', txt.strip())) for t, txt in gold_entities]
            pred_normalized = [(t, re.sub(r'\s+', ' ', txt.strip())) for t, txt in pred_entities]

        gold_counts = Counter(gold_normalized)
        pred_counts = Counter(pred_normalized)

        for entity, g_count in gold_counts.items():
            p_count = pred_counts.get(entity, 0)
            tp = min(g_count, p_count)
            fn = g_count - tp
            tp_total += tp
            fn_total += fn
            ent_type = entity[0]
            if ent_type not in type_stats:
                type_stats[ent_type] = {'tp': 0, 'fp': 0, 'fn': 0}
            type_stats[ent_type]['tp'] += tp
            type_stats[ent_type]['fn'] += fn

        for entity, p_count in pred_counts.items():
            g_count = gold_counts.get(entity, 0)
            if p_count > g_count:
                fp = p_count - g_count
                fp_total += fp
                ent_type = entity[0]
                if ent_type not in type_stats:
                    type_stats[ent_type] = {'tp': 0, 'fp': 0, 'fn': 0}
                type_stats[ent_type]['fp'] += fp

    total_gold = tp_total + fn_total
    total_pred = tp_total + fp_total

    # FIX: empty gold + empty pred = perfect performance
    if total_gold == 0 and total_pred == 0:
        return {
            "overall": {
                "precision": 1.0, "recall": 1.0, "f1": 1.0,
                "tp": 0, "gold_count": 0, "pred_count": 0
            },
            "per_type": {}
        }

    precision = tp_total / total_pred if total_pred > 0 else 0.0
    recall    = tp_total / total_gold if total_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    results = {
        "overall": {
            "precision": precision, "recall": recall, "f1": f1,
            "tp": tp_total, "gold_count": total_gold, "pred_count": total_pred
        },
        "per_type": {}
    }

    for ent_type in sorted(type_stats.keys()):
        stats = type_stats[ent_type]
        t_tp, t_fp, t_fn = stats['tp'], stats['fp'], stats['fn']
        t_gold = t_tp + t_fn
        t_pred = t_tp + t_fp
        p = t_tp / t_pred if t_pred > 0 else 0.0
        r = t_tp / t_gold if t_gold > 0 else 0.0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        results["per_type"][ent_type] = {
            "precision": p, "recall": r, "f1": f,
            "tp": t_tp, "gold_count": t_gold, "pred_count": t_pred
        }

    return results


def print_metrics(results, title="NER Evaluation Results"):
    """Pretty-print evaluation results."""
    print(f"\n{'=' * 60}")
    print(title)
    print(f"{'=' * 60}")

    o = results["overall"]
    print(f"\nOverall:  P={o['precision']:.3f}  R={o['recall']:.3f}  F1={o['f1']:.3f}")
    print(f"          ({o['tp']} TP / {o['gold_count']} gold / {o['pred_count']} predicted)\n")

    print(f"{'Type':>8s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'TP':>4s}  {'Gold':>5s}  {'Pred':>5s}")
    print(f"{'-' * 52}")

    # Виводимо класи за алфавітом
    for ent_type, m in sorted(results["per_type"].items()):
        print(f"{ent_type:>8s}  {m['precision']:6.3f}  {m['recall']:6.3f}  {m['f1']:6.3f}  "
              f"{m['tp']:4d}  {m['gold_count']:5d}  {m['pred_count']:5d}")

In [ ]:
print("TEST 1: Perfect round-trip (parser extracts what we inserted)")

sample = train_chunks[0]
gold = parse_inline_tagged_output(sample["tagged_text"])
pred = parse_inline_tagged_output(sample["tagged_text"])
results = compute_ner_metrics(gold, pred)
print_metrics(results, f"Perfect match test: {sample['name']}")

TEST 1: Perfect round-trip (parser extracts what we inserted)

Perfect match test: 0124bf2fdf35_chunk_0

Overall:  P=1.000  R=1.000  F1=1.000
          (6 TP / 6 gold / 6 predicted)

    Type    Prec     Rec      F1    TP   Gold   Pred
----------------------------------------------------
    PERS   1.000   1.000   1.000     6      6      6


In [ ]:
print("TEST 2: Simulated imperfect prediction")

# Simulate model output with inline tags
gold_tagged = "[PERS: Олена Петренко] працює у [ORG: Київстар] у [LOC: Київ] з [DATE: 29 квітня] за [MON: 7,08 млн грн]."
pred_tagged = "[PERS: Олена Петренко] працює у [ORG: Київстар] у [LOC: Київ] з 29 квітня за 7,08 млн грн. [PERS: Київстар] — невідома особа."

gold_example = parse_inline_tagged_output(gold_tagged)
pred_example = parse_inline_tagged_output(pred_tagged)

print(f"Gold entities: {gold_example}")
print(f"Pred entities: {pred_example}")

results = compute_ner_metrics(gold_example, pred_example)
print_metrics(results, "Simulated imperfect prediction")

TEST 2: Simulated imperfect prediction
Gold entities: [('PERS', 'Олена Петренко'), ('ORG', 'Київстар'), ('LOC', 'Київ'), ('DATE', '29 квітня'), ('MON', '7,08 млн грн')]
Pred entities: [('PERS', 'Олена Петренко'), ('ORG', 'Київстар'), ('LOC', 'Київ'), ('PERS', 'Київстар')]

Simulated imperfect prediction

Overall:  P=0.750  R=0.600  F1=0.667
          (3 TP / 5 gold / 4 predicted)

    Type    Prec     Rec      F1    TP   Gold   Pred
----------------------------------------------------
    DATE   0.000   0.000   0.000     0      1      0
     LOC   1.000   1.000   1.000     1      1      1
     MON   0.000   0.000   0.000     0      1      0
     ORG   1.000   1.000   1.000     1      1      1
    PERS   0.500   1.000   0.667     1      1      2


In [ ]:
print("TEST 3: Full round-trip on all train chunks")

all_gold = []
all_pred = []
parse_failures = 0

for c in train_chunks:
    gold = parse_inline_tagged_output(c["tagged_text"])
    if len(gold) != c["num_entities"]:
        parse_failures += 1
        if parse_failures <= 5:  # show first 5 mismatches for debugging
            print(f"  MISMATCH: {c['name']} — expected {c['num_entities']}, parsed {len(gold)}")
            print(f"    Tagged: {(c['tagged_text'])}")
    all_gold.extend(gold)
    all_pred.extend(gold)  # perfect match: pred = gold

results = compute_ner_metrics(all_gold, all_pred)
print_metrics(results, "Full train set round-trip")
print(f"\nParse mismatches: {parse_failures} out of {len(train_chunks)}")

TEST 3: Full round-trip on all train chunks

Full train set round-trip

Overall:  P=1.000  R=1.000  F1=1.000
          (17288 TP / 17288 gold / 17288 predicted)

    Type    Prec     Rec      F1    TP   Gold   Pred
----------------------------------------------------
     ART   1.000   1.000   1.000   448    448    448
    DATE   1.000   1.000   1.000  1412   1412   1412
     DOC   1.000   1.000   1.000   113    113    113
     JOB   1.000   1.000   1.000  1558   1558   1558
     LOC   1.000   1.000   1.000  2515   2515   2515
    MISC   1.000   1.000   1.000   429    429    429
     MON   1.000   1.000   1.000   746    746    746
     ORG   1.000   1.000   1.000  4193   4193   4193
     PCT   1.000   1.000   1.000   205    205    205
  PERIOD   1.000   1.000   1.000   473    473    473
    PERS   1.000   1.000   1.000  4832   4832   4832
   QUANT   1.000   1.000   1.000   335    335    335
    TIME   1.000   1.000   1.000    29     29     29

Parse mismatches: 0 out of 1667


# Eval pipeline

In [ ]:
def run_ner_inference(model, tokenizer, text, max_new_tokens=None):
    """Run NER on a single text."""
    messages = create_chat_messages(text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Inline tagging: output ≈ input text + ~20% tag overhead
    if max_new_tokens is None:
        input_text_tokens = len(tokenizer(text, add_special_tokens=False).input_ids)
        max_new_tokens = int(input_text_tokens * 1.3) + 50

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # Forces greedy decoding
            temperature=None,       # Explicitly clears temperature
            top_p=None,             # Clears warning
            top_k=None,             # Clears warning
            repetition_penalty=1.0  # CRITICAL: 1.0 means no penalty!
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    return response

In [ ]:
from tqdm.auto import tqdm

def run_ner_inference_batch(model, tokenizer, texts, max_new_tokens=None,
                             batch_size=4, progress_bar=None,
                             on_batch_done=None):
    all_responses = []

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    own_bar = progress_bar is None
    if own_bar:
        progress_bar = tqdm(
            total=len(texts),
            desc="Inference",
            unit="chunk",
            ncols=100,
            leave=True
        )

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_indices = list(range(i, min(i + batch_size, len(texts))))

        prompts = []
        for text in batch_texts:
            messages = create_chat_messages(text)
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(prompt)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=3072
        ).to(model.device)

        if max_new_tokens is None:
            longest_text = max(batch_texts, key=len)
            text_tokens = len(tokenizer(longest_text, add_special_tokens=False).input_ids)
            batch_max_tokens = int(text_tokens * 1.3) + 50
        else:
            batch_max_tokens = max_new_tokens

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=batch_max_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                repetition_penalty=1.0,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

        # FIX: use full padded input length, not non-padding count
        padded_prompt_length = inputs.input_ids.shape[1]

        batch_responses = []
        for j, output in enumerate(outputs):
            response = tokenizer.decode(
                output[padded_prompt_length:],
                skip_special_tokens=True
            )
            batch_responses.append(response)
            all_responses.append(response)

        progress_bar.update(len(batch_texts))

        if on_batch_done is not None:
            on_batch_done(batch_responses, batch_indices)

    if own_bar:
        progress_bar.close()

    return all_responses


In [ ]:
import json

def evaluate_on_chunks_fast(model, tokenizer, chunks, batch_size=4, max_examples=None,
                             hard_chunks_file=None, hard_f1_threshold=0.5):
    """Fast batched evaluation on chunks with live F1 tracking.

    Args:
        hard_chunks_file:    Path to JSONL file to write hard chunks into.
                             Written immediately when found - safe to interrupt.
        hard_f1_threshold:   Chunks with per-chunk F1 below this are saved.
    """
    if max_examples:
        chunks = chunks[:max_examples]

    texts = [c["input_text"] for c in chunks]

    # Running accumulators for live metrics
    all_gold = []
    all_pred = []

    # Open hard chunks file once (append mode - safe to resume)
    hard_file = open(hard_chunks_file, "a", encoding="utf-8") if hard_chunks_file else None

    # Create progress bar with postfix for live metrics
    progress_bar = tqdm(
        total=len(texts),
        desc=f"Evaluating ({len(texts)} chunks)",
        unit="chunk",
        ncols=120,
        leave=True
    )

    def on_batch_done(batch_responses, batch_indices):
        """Called after each batch — compute running metrics."""
        for idx, response in zip(batch_indices, batch_responses):
            gold = parse_inline_tagged_output(chunks[idx]["tagged_text"])
            pred = parse_inline_tagged_output(response)
            all_gold.append(gold)
            all_pred.append(pred)

            # --- Per-chunk F1 for hard example detection ---
            if hard_file is not None:
                chunk_metrics = compute_ner_metrics([gold], [pred])
                chunk_f1 = chunk_metrics["overall"]["f1"]

                if chunk_f1 < hard_f1_threshold:
                    record = {
                        "chunk_idx":     idx,
                        "chunk_name":    chunks[idx].get("name", f"chunk_{idx}"),
                        "original_name": chunks[idx].get("original_name", ""),
                        "subcorpus":     chunks[idx].get("subcorpus", ""),
                        "chunk_f1":      round(chunk_f1, 4),
                        "gold_count":    chunk_metrics["overall"]["gold_count"],
                        "pred_count":    chunk_metrics["overall"]["pred_count"],
                        "tp":            chunk_metrics["overall"]["tp"],
                        "per_type":      {
                            t: round(m["f1"], 4)
                            for t, m in chunk_metrics["per_type"].items()
                        },
                        # Full content for later analysis
                        "input_text":    chunks[idx]["input_text"],
                        "gold_tagged":   chunks[idx]["tagged_text"],
                        "pred_tagged":   response,
                    }
                    hard_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                    hard_file.flush()  # Write immediately - don't buffer

        # Compute running metrics
        if len(all_gold) > 0:
            running = compute_ner_metrics(all_gold, all_pred)
            overall = running["overall"]
            per_type = running["per_type"]

            if per_type:
                macro_f1 = sum(m["f1"] for m in per_type.values()) / len(per_type)
            else:
                macro_f1 = 0.0

            progress_bar.set_postfix({
                "micro_F1": f"{overall['f1']:.3f}",
                "macro_F1": f"{macro_f1:.3f}",
                "P": f"{overall['precision']:.3f}",
                "R": f"{overall['recall']:.3f}",
                "gold": overall["gold_count"],
                "pred": overall["pred_count"],
            })

    try:
        run_ner_inference_batch(
            model, tokenizer, texts,
            batch_size=batch_size,
            progress_bar=progress_bar,
            on_batch_done=on_batch_done
        )
    finally:
        # Always close file even if evaluation crashes or is interrupted
        if hard_file is not None:
            hard_file.close()

    progress_bar.close()

    results = compute_ner_metrics(all_gold, all_pred)
    return results

# Test eval pipeline

In [ ]:
sample = test_chunks[0]
messages = create_chat_messages(sample["input_text"])
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_tokens = len(tokenizer(prompt, add_special_tokens=False).input_ids)

import time
start = time.time()
response = run_ner_inference(model, tokenizer, sample["input_text"])
elapsed = time.time() - start

In [ ]:
sample["input_text"]

'Вивчали хімічний режим підземних вод, які використовуються для напування тварин у західній біогеохімічній зоні України.\n\nЗа органолептичними показниками та хімічним складом вони відповідають чинним вимогам.\n\nПроте у воді досліджуваних господарств виявили меркурій, що досить небезпечно, а вміст феруму та мангану перевищував нормативні величини втричі.\n\nВода - важлива складова всього живого на Землі.\n\nВона входить до складу органів і тканин тварин.\n\nВода становить у середньому 75% маси тіла ссавців.\n\nУсі біохімічні реакції, що лежать в основі життя, відбуваються за її безпосередньої участі.\n\nВода в крові й тканинній рідині бере участь у транспорті речовин між клітинами й органами, регулює осмотичний тиск, підтримує температурний гомеостаз, виводить продукти обміну з організму.\n\nХімічний склад води здебільшого свідчить про хімічний склад середовища, в якому вона перебуває.\n\nЯк надлишок, так і нестача мінеральних солей у водному середовищі (зокрема мікроелементів) дають 

In [ ]:
response_tokens = len(tokenizer(response, add_special_tokens=False).input_ids)
print(f"Input tokens:    {input_tokens}")
print(f"Response tokens: {response_tokens}")
print(f"Time:            {elapsed:.1f}s")
print(f"Tokens/second:   {response_tokens/elapsed:.1f}")
print(f"{response}")

Input tokens:    915
Response tokens: 349
Time:            70.8s
Tokens/second:   4.9
Вивчали хімічний режим підземних вод, які використовуються для напування тварин у західній біогеохімічній зоні [LOC: України].

За органолептичними показниками та хімічним складом вони відповідають чинним вимогам.

Проте у воді досліджуваних господарств виявили меркурій, що досить небезпечно, а вміст феруму та мангану перевищував нормативні величини втричі.

Вода - важлива складова всього живого на [LOC: Землі].

Вона входить до складу органів і тканин тварин.

Вода становить у середньому [PCT: 75%] маси тіла ссавців.

Усі біохімічні реакції, що лежать в основі життя, відбуваються за її безпосередньої участі.

Вода в крові й тканинній рідині бере участь у транспорті речовин між клітинами й органами, регулює осмотичний тиск, підтримує температурний гомеостаз, виводить продукти обміну з організму.

Хімічний склад води здебільшого свідчить про хімічний склад середовища, в якому вона перебуває.

Як надлиш

# Run Inference

In [ ]:
# print("=" * 60)
# print("FULL TEST SET EVALUATION")
# print("=" * 60)

# start = time.time()
# results_full = evaluate_on_chunks_fast(model, tokenizer, test_chunks, batch_size=4)
# elapsed = time.time() - start

# print_metrics(results_full, "Full test set")
# print(f"\nTotal time: {elapsed/60:.1f} minutes")
# print(f"Chunks evaluated: {len(test_chunks)}")

In [ ]:
import random
import time
random.seed(42)

# SUBSET_SIZE = 128
# eval_subset = random.sample(test_chunks, min(SUBSET_SIZE, len(test_chunks)))
eval_subset = test_chunks
total_gold_in_subset = sum(c["num_entities"] for c in eval_subset)

print("=" * 60)
print(f"SUBSET EVALUATION ({len(eval_subset)} chunks, ~{total_gold_in_subset} entities)")
print("=" * 60)

start = time.time()
results_subset = evaluate_on_chunks_fast(model, tokenizer, eval_subset, batch_size=12,
                                         hard_chunks_file="hard_chunks_mamay_9b_gepa_prompt.jsonl",
                                         hard_f1_threshold=0.7)
elapsed = time.time() - start

print_metrics(results_subset, f"Subset ({len(eval_subset)} chunks)")
print(f"\nTotal time: {elapsed/60:.1f} minutes")

SUBSET EVALUATION (653 chunks, ~6806 entities)


Evaluating (653 chunks):   0%|                                                               | 0/653 [00:00<?,…


Subset (653 chunks)

Overall:  P=0.593  R=0.524  F1=0.557
          (3569 TP / 6806 gold / 6019 predicted)

    Type    Prec     Rec      F1    TP   Gold   Pred
----------------------------------------------------
     ART   0.463   0.621   0.531   146    235    315
    DATE   0.498   0.225   0.310   113    503    227
     DOC   0.114   0.231   0.153     9     39     79
     JOB   0.455   0.030   0.057    20    657     44
     LOC   0.550   0.528   0.539   431    817    783
    MISC   0.046   0.246   0.078    35    142    761
     MON   0.944   0.728   0.822   236    324    250
     ORG   0.804   0.654   0.721  1004   1536   1248
     PCT   1.000   0.056   0.105     5     90      5
  PERIOD   0.706   0.065   0.119    12    184     17
    PERS   0.698   0.691   0.694  1506   2180   2159
   QUANT   0.417   0.562   0.478    50     89    120
    TIME   0.182   0.200   0.190     2     10     11

Total time: 183.5 minutes


# Fine-Tuning pipeline

In [ ]:
def prepare_training_dataset(chunks, tokenizer):
    """
    Convert chunks into the chat format expected by SFTTrainer.
    Returns a HuggingFace Dataset object.
    """
    from datasets import Dataset
    import random

    conversations = []
    for chunk in chunks:
        if chunk["num_entities"] == 0:
            if random.random() > 0.1:
                continue

        messages = create_training_messages(chunk["input_text"], chunk["tagged_text"])
        formatted = tokenizer.apply_chat_template(messages, tokenize=False)
        conversations.append({"text": formatted})

    print(f"Training examples: {len(conversations)} (from {len(chunks)} chunks)")
    print(f"  Skipped empty chunks: {len(chunks) - len(conversations)}")
    return Dataset.from_list(conversations)

In [ ]:
train_dataset = prepare_training_dataset(train_chunks, tokenizer)
print(f"Dataset ready: {train_dataset}")
print(f"Example preview (first 200 chars):\n{train_dataset[0]['text'][:200]}...")

Training examples: 1555 (from 1655 chunks)
  Skipped empty chunks: 100
Dataset ready: Dataset({
    features: ['text'],
    num_rows: 1555
})
Example preview (first 200 chars):
<bos><start_of_turn>user
Ти — інструмент точної розмітки тексту, а не редактор. Перепиши текст, позначивши іменовані сутності у форматі [ТИП: сутність].

Типи:
PERS — імена людей, літературних персона...


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()

trainable params: 216,072,192 || all params: 9,457,778,176 || trainable%: 2.2846


In [ ]:
import torch
import torch.nn.functional as F
from trl import SFTTrainer, SFTConfig

class NERWeightedSFTTrainer(SFTTrainer):
    def __init__(self, *args, boundary_weight=4.0, base_tag_weight=1.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_weights = {}

        # Accessing the tokenizer correctly for Mamay 9B (Gemma 2)
        tk = self.processing_class

        # 1. Boundary Tokens
        bracket_open_ids = set(
            tk.encode("[", add_special_tokens=False) +
            tk.encode(" [", add_special_tokens=False)
        )
        for tid in bracket_open_ids:
            self.token_weights[tid] = boundary_weight

        # 2. Structural End/Colon Tokens
        struct_ids = set(
            tk.encode(":", add_special_tokens=False) +
            tk.encode("]", add_special_tokens=False)
        )
        for tid in struct_ids:
            self.token_weights[tid] = base_tag_weight

        # 3. Granular Class Weights
        weight_scheme = {
            1.2: ["PERS", "ORG", "LOC"],
            2.5: ["DATE", "JOB", "MON", "ART", "PERIOD"],
            6.0: ["MISC", "QUANT", "PCT", "DOC", "TIME"]
        }

        for weight, tags in weight_scheme.items():
            for tag in tags:
                ids = tk.encode(tag, add_special_tokens=False)
                for tid in ids:
                    self.token_weights[tid] = weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction='none'
        )

        weights = torch.ones_like(shift_labels, dtype=torch.float, device=shift_labels.device)
        for tid, weight_val in self.token_weights.items():
            weights[shift_labels == tid] = weight_val

        weights[shift_labels == -100] = 0.0
        loss = (loss_fct * weights.view(-1)).sum() / weights.view(-1).sum()

        return (loss, outputs) if return_outputs else loss

In [ ]:

sft_config = SFTConfig(
    output_dir="/content/checkpoints/mamay-9b-ner-v1",

    # --- Training duration ---
    num_train_epochs=8,

    # --- Batch size ---
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # --- Learning rate ---
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # --- Memory optimization ---
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # --- SFT-specific ---
    max_length=1536,
    packing=False,
    dataset_text_field="text",

    # --- Logging ---
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=10,

    # --- Misc ---
    max_grad_norm=1.0,
    report_to="none",
)

trainer = NERWeightedSFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    boundary_weight=4.0,
    base_tag_weight=1.5,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1555 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1555 [00:00<?, ? examples/s]

In [ ]:
print(f"\nTraining config summary:")
print(f"  Total examples:     {len(train_dataset)}")
print(f"  Effective batch:    {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Max seq length:     {sft_config.max_length}")
print(f"  Max steps (test):   {sft_config.max_steps}")


Training config summary:
  Total examples:     1555
  Effective batch:    8
  Max seq length:     1536
  Max steps (test):   -1


# Train Model

In [ ]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

1355

In [ ]:
print("\n" + "=" * 60)
print("Training...")
print("=" * 60)

# trainer.train()
trainer.train(resume_from_checkpoint="/content/checkpoints/mamay-9b-ner-v1/checkpoint-582")

print("\nTraining pipeline verified successfully!")


Training...


Step,Training Loss
590,1.001016
600,0.877930
610,0.937924
620,0.949478
630,0.891473
640,0.931266
650,0.927727
660,0.912949
670,0.983854
680,0.956239



Training pipeline verified successfully!


In [ ]:
print("Saving LoRA adapter...")
trainer.model.save_pretrained("./ner-mamay-9b-8e")
tokenizer.save_pretrained("./ner-mamay-9b-8e")
print("Adapter saved!")

Saving LoRA adapter...
Adapter saved!


In [ ]:
import shutil

# Create a zip archive from the saved adapter folder
shutil.make_archive("./ner-mamay-9b-8e", "zip", "./ner-mamay-9b-8e")
print("Archive created: ner-mamay-9b-8e.zip")

Archive created: ner-mamay-9b-8e.zip


# Evaluate fine-tuned model

In [ ]:

import torch, gc
torch.cuda.empty_cache()
gc.collect()


113

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, "/content/checkpoints/mamay-9b-ner-v1/checkpoint-780")

In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [ ]:
import random
random.seed(42)
# eval_subset = random.sample(test_chunks, min(52, len(test_chunks)))
eval_subset = test_chunks
total_gold = sum(c["num_entities"] for c in eval_subset)

print(f"\n{'=' * 60}")
print(f"FINE-TUNED EVALUATION ({len(eval_subset)} chunks, ~{total_gold} entities)")
print(f"{'=' * 60}")

import time
start = time.time()
results_finetuned = evaluate_on_chunks_fast(model, tokenizer, eval_subset, batch_size=8,
                                            hard_chunks_file="hard_chunks_mamay9b_epoch4.jsonl",
                                            hard_f1_threshold=0.7)
elapsed = time.time() - start

print_metrics(results_finetuned, "MamayLM + QLoRA Fine-tuned")
print(f"\nTotal time: {elapsed/60:.1f} minutes")


FINE-TUNED EVALUATION (647 chunks, ~6807 entities)


Evaluating (647 chunks):   0%|                                                               | 0/647 [00:00<?,…


MamayLM + QLoRA Fine-tuned

Overall:  P=0.884  R=0.870  F1=0.877
          (5923 TP / 6807 gold / 6699 predicted)

    Type    Prec     Rec      F1    TP   Gold   Pred
----------------------------------------------------
     ART   0.756   0.779   0.767   183    235    242
    DATE   0.891   0.875   0.883   440    503    494
     DOC   0.485   0.400   0.438    16     40     33
     JOB   0.706   0.626   0.663   411    657    582
     LOC   0.900   0.896   0.898   732    817    813
    MISC   0.403   0.394   0.399    56    142    139
     MON   0.991   0.966   0.978   313    324    316
     ORG   0.908   0.907   0.907  1393   1536   1534
     PCT   0.896   0.956   0.925    86     90     96
  PERIOD   0.727   0.652   0.688   120    184    165
    PERS   0.954   0.957   0.955  2086   2180   2187
   QUANT   0.900   0.910   0.905    81     89     90
    TIME   0.750   0.600   0.667     6     10      8

Total time: 170.7 minutes


In [ ]:
# === Step 5: Print comparison with zero-shot baseline ===
# NOTE: For a fair comparison, re-run zero-shot evaluation with the same
# inline tagging format and the same test chunks before finalizing these numbers.

print(f"\n{'=' * 60}")
print("COMPARISON: Zero-Shot vs Fine-Tuned")
print(f"{'=' * 60}")

# Zero-shot results — UPDATE these after re-running zero-shot with inline format
zs = {"P": 0.282, "R": 0.067, "F1": 0.108}
ft = results_finetuned["overall"]

print(f"\n{'':>12s}  {'Zero-Shot':>10s}  {'Fine-Tuned':>10s}  {'Change':>10s}")
print(f"{'-' * 48}")
print(f"{'Precision':>12s}  {zs['P']:10.3f}  {ft['precision']:10.3f}  {ft['precision']-zs['P']:+10.3f}")
print(f"{'Recall':>12s}  {zs['R']:10.3f}  {ft['recall']:10.3f}  {ft['recall']-zs['R']:+10.3f}")
print(f"{'F1':>12s}  {zs['F1']:10.3f}  {ft['f1']:10.3f}  {ft['f1']-zs['F1']:+10.3f}")

# Per-type comparison
print(f"\n{'':>8s}  {'ZS F1':>6s}  {'FT F1':>6s}  {'Change':>8s}")
print(f"{'-' * 35}")

zs_per_type = {
    "ART": 0.038, "DATE": 0.192, "JOB": 0.024, "LOC": 0.086,
    "MON": 0.066, "ORG": 0.192, "PERIOD": 0.033, "PERS": 0.053
}

for ent_type in sorted(zs_per_type.keys()):
    zs_f1 = zs_per_type[ent_type]
    ft_f1 = results_finetuned["per_type"].get(ent_type, {}).get("f1", 0.0)
    print(f"{ent_type:>8s}  {zs_f1:6.3f}  {ft_f1:6.3f}  {ft_f1-zs_f1:+8.3f}")


COMPARISON: Zero-Shot vs Fine-Tuned

               Zero-Shot  Fine-Tuned      Change
------------------------------------------------
   Precision       0.282       0.873      +0.591
      Recall       0.067       0.853      +0.786
          F1       0.108       0.863      +0.755

           ZS F1   FT F1    Change
-----------------------------------
     ART   0.038   0.769    +0.731
    DATE   0.192   0.887    +0.695
     JOB   0.024   0.685    +0.661
     LOC   0.086   0.908    +0.822
     MON   0.066   0.854    +0.788
     ORG   0.192   0.895    +0.703
  PERIOD   0.033   0.690    +0.657
    PERS   0.053   0.941    +0.888


# Visualize results

In [ ]:
def visualize_ner_example(model, tokenizer, chunk, show_raw=True):
    """Run NER on a single test chunk and display results side-by-side with gold."""
    text = chunk["input_text"]
    gold_tagged = chunk["tagged_text"]

    # Run inference
    response = run_ner_inference(model, tokenizer, text)

    # Parse both
    gold_entities = parse_inline_tagged_output(gold_tagged)
    pred_entities = parse_inline_tagged_output(response)

    gold_set = set(gold_entities)
    pred_set = set(pred_entities)

    tp = gold_set & pred_set
    fn = gold_set - pred_set  # missed
    fp = pred_set - gold_set  # hallucinated

    # Text preservation check
    stripped_response = re.sub(r'\[([A-Za-z]+):\s*(.+?)\]', r'\2', response, flags=re.DOTALL)
    stripped_normalized = re.sub(r'\s+', ' ', stripped_response).strip()
    orig_normalized = re.sub(r'\s+', ' ', text).strip()
    from difflib import SequenceMatcher
    preservation = SequenceMatcher(None, orig_normalized.split(), stripped_normalized.split()).ratio()

    # Display
    print("=" * 70)
    print("INPUT TEXT (first 500 chars):")
    print("=" * 70)
    print(text)
    if len(text) > 500:
        print(f"... ({len(text)} chars total)")

    print("\n" + "=" * 70)
    print(f"GOLD ENTITIES ({len(gold_entities)}):")
    print("=" * 70)
    for etype, etext in sorted(gold_entities):
        marker = "  ✓" if (etype, etext) in tp else "  ✗ MISSED"
        print(f"  {etype:8s} | {etext}{marker}")

    print(f"\nPREDICTED ENTITIES ({len(pred_entities)}):")
    print("-" * 70)
    for etype, etext in sorted(pred_entities):
        marker = "  ✓" if (etype, etext) in tp else "  ✗ HALLUCINATED"
        print(f"  {etype:8s} | {etext}{marker}")

    print(f"\nSCORE: {len(tp)} TP / {len(fn)} missed / {len(fp)} hallucinated")
    print(f"TEXT PRESERVATION: {preservation:.1%}")

    if show_raw:
        print(f"\n{'=' * 70}")
        print("RAW MODEL OUTPUT (first 500 chars):")
        print("-" * 70)
        print(response)
        if len(response) > 500:
            print(f"... ({len(response)} chars total)")

    print("=" * 70)
    return {"gold": gold_entities, "pred": pred_entities, "tp": tp, "fn": fn, "fp": fp,
            "preservation": preservation}

In [ ]:
# Pick a few random test chunks with entities
import random
random.seed(42)

non_empty = [c for c in test_chunks if c["num_entities"] > 0]
samples = random.sample(non_empty, min(10, len(non_empty)))

for i, chunk in enumerate(samples):
    print(f"\n{'#' * 70}")
    print(f"  EXAMPLE {i+1} / {len(samples)}  —  source: {chunk.get('original_name', 'unknown')}")
    print(f"{'#' * 70}")
    visualize_ner_example(model, tokenizer, chunk)
    print("\n")


######################################################################
  EXAMPLE 1 / 10  —  source: fa2b54229991
######################################################################
[DATE: 12 листопада] до [ORG: Окружного адміністративного суду Києва] надійшов позов від «[ORG: Укрнафти]» проти [ORG: АМКУ] - із вимогами вилучити і знищити усі докази у справі, отримані від [ORG: НАБУ].
Суд відмовив у відкритті провадження, посилаючись на те, що спір належить розглядати у юрисдикції господарського суду.
Втім, не погодившись із такою позицією суду, «[ORG: Укрнафта]» успішно оскаржила її в апеляційній інстанції.
На засіданні [DATE: 13 листопада 2018 року] [ORG: АМКУ] мав прийняти кінцеві висновки у справі, однак незадовго до цього, [DATE: 8 листопада], «приватівці» заблокували через суд розгляд цього провадження.
[DATE: 4 грудня], коли [ORG: АМКУ] зробив спробу розглянути справу, на засіданні з'явився [JOB: держвиконавець] із нагадуванням про судову заборону.
Нагадаємо, з [DATE: 2003 рок

KeyboardInterrupt: 

# Hard test chunks analysis

In [ ]:
import json
from collections import Counter, defaultdict
import re

def analyze_hard_chunks(filepath):
    """Comprehensive analysis of hard chunks JSONL file."""

    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    print(f"Total hard chunks (F1 < threshold): {len(records)}")
    print(f"Subcorpora present: {sorted(set(r['subcorpus'] for r in records))}")

    # ── 1. F1 DISTRIBUTION ──────────────────────────────────────────
    print("\n" + "=" * 60)
    print("1. F1 DISTRIBUTION")
    print("=" * 60)

    buckets = {"0.0 (zero)": 0, "0.0–0.3": 0, "0.3–0.5": 0, "0.5–0.7": 0}
    for r in records:
        f = r["chunk_f1"]
        if f == 0.0:        buckets["0.0 (zero)"] += 1
        elif f < 0.3:       buckets["0.0–0.3"] += 1
        elif f < 0.5:       buckets["0.3–0.5"] += 1
        else:               buckets["0.5–0.7"] += 1

    for bucket, count in buckets.items():
        bar = "█" * count
        print(f"  {bucket:<12}: {count:>3}  {bar}")

    # ── 2. PER-SUBCORPUS BREAKDOWN ───────────────────────────────────
    print("\n" + "=" * 60)
    print("2. HARD CHUNKS PER SUBCORPUS")
    print("=" * 60)

    sub_stats = defaultdict(lambda: {"count": 0, "f1_sum": 0.0, "zero_f1": 0})
    for r in records:
        s = r["subcorpus"]
        sub_stats[s]["count"] += 1
        sub_stats[s]["f1_sum"] += r["chunk_f1"]
        if r["chunk_f1"] == 0.0:
            sub_stats[s]["zero_f1"] += 1

    print(f"  {'Subcorpus':<15}  {'Hard chunks':>11}  {'Avg F1':>7}  {'Zero F1':>8}")
    print("  " + "-" * 45)
    for sub, st in sorted(sub_stats.items(), key=lambda x: -x[1]["count"]):
        avg_f1 = st["f1_sum"] / st["count"]
        print(f"  {sub:<15}  {st['count']:>11}  {avg_f1:>7.3f}  {st['zero_f1']:>8}")

    # ── 3. ENTITY TYPE ERROR ANALYSIS ────────────────────────────────
    print("\n" + "=" * 60)
    print("3. ENTITY TYPE ERRORS (from per_type in hard chunks)")
    print("=" * 60)

    # Count how often each type appears with low F1 in hard chunks
    type_zero   = Counter()  # F1 = 0.0 for this type
    type_low    = Counter()  # F1 < 0.5 for this type
    type_seen   = Counter()  # Type appeared at all in these chunks

    for r in records:
        for t, f1 in r["per_type"].items():
            type_seen[t] += 1
            if f1 == 0.0:   type_zero[t] += 1
            if f1 < 0.5:    type_low[t]  += 1

    print(f"  {'Type':<8}  {'Seen':>5}  {'F1=0':>6}  {'F1<0.5':>7}  {'Zero%':>6}")
    print("  " + "-" * 40)
    for t, seen in type_seen.most_common():
        print(f"  {t:<8}  {seen:>5}  {type_zero[t]:>6}  {type_low[t]:>7}  "
              f"{type_zero[t]/seen*100:>5.0f}%")

    # ── 4. FALSE POSITIVES / FALSE NEGATIVES ─────────────────────────
    print("\n" + "=" * 60)
    print("4. FALSE POSITIVES & FALSE NEGATIVES")
    print("=" * 60)

    if "false_positives" in records[0]:
        fp_types = Counter()
        fn_types = Counter()
        fp_texts = Counter()
        fn_texts = Counter()

        for r in records:
            for t, txt in r.get("false_positives", []):
                fp_types[t] += 1
                fp_texts[(t, txt[:40])] += 1
            for t, txt in r.get("false_negatives", []):
                fn_types[t] += 1
                fn_texts[(t, txt[:40])] += 1

        print("  FALSE POSITIVES (hallucinated) by type:")
        for t, count in fp_types.most_common():
            print(f"    {t:<8}: {count}")

        print("\n  FALSE NEGATIVES (missed) by type:")
        for t, count in fn_types.most_common():
            print(f"    {t:<8}: {count}")

        print("\n  Top 10 most hallucinated entities:")
        for (t, txt), count in fp_texts.most_common(10):
            print(f"    [{t}] '{txt}' × {count}")

        print("\n  Top 10 most missed entities:")
        for (t, txt), count in fn_texts.most_common(10):
            print(f"    [{t}] '{txt}' × {count}")
    else:
        # Compute FP/FN from gold/pred tagged text on the fly
        print("  (fp/fn not stored — computing from gold/pred text)")

        fp_types = Counter()
        fn_types = Counter()

        for r in records:
            gold = parse_inline_tagged_output(r["gold_tagged"])
            pred = parse_inline_tagged_output(r["pred_tagged"])

            from collections import Counter as C
            g = C(gold)
            p = C(pred)

            for ent, cnt in g.items():
                missed = cnt - p.get(ent, 0)
                fn_types[ent[0]] += max(missed, 0)

            for ent, cnt in p.items():
                halluc = cnt - g.get(ent, 0)
                fp_types[ent[0]] += max(halluc, 0)

        print("  Missed by type (FN):")
        for t, count in fn_types.most_common():
            print(f"    {t:<8}: {count}")

        print("\n  Hallucinated by type (FP):")
        for t, count in fp_types.most_common():
            print(f"    {t:<8}: {count}")

    # ── 5. ERROR TYPE CLASSIFICATION ─────────────────────────────────
    print("\n" + "=" * 60)
    print("5. ERROR PATTERN CLASSIFICATION")
    print("=" * 60)

    wrong_type   = 0  # Entity text found in pred but with WRONG type
    missed_ent   = 0  # Entity completely absent from pred
    hallucinated = 0  # Pred entity not in gold at all
    boundary_err = 0  # Partial text match (substring of gold entity)

    for r in records:
        gold = parse_inline_tagged_output(r["gold_tagged"])
        pred = parse_inline_tagged_output(r["pred_tagged"])

        gold_texts = {txt.lower(): typ for typ, txt in gold}
        pred_texts = {txt.lower(): typ for typ, txt in pred}

        # Wrong type: same text, different label
        for txt, g_type in gold_texts.items():
            if txt in pred_texts and pred_texts[txt] != g_type:
                wrong_type += 1

        # Missed: gold text not in pred at all (even partially)
        for typ, txt in gold:
            if (typ, txt) not in pred:
                # Check if it's a boundary error (substring match)
                is_substring = any(
                    txt.lower() in p_txt.lower() or p_txt.lower() in txt.lower()
                    for _, p_txt in pred
                    if abs(len(txt) - len(p_txt)) < len(txt) * 0.5
                )
                if is_substring:
                    boundary_err += 1
                else:
                    missed_ent += 1

        # Hallucinated: pred entity not in gold
        for typ, txt in pred:
            if (typ, txt) not in gold:
                hallucinated += 1

    total_errors = wrong_type + missed_ent + hallucinated + boundary_err
    print(f"  {'Error type':<25}  {'Count':>6}  {'Share':>6}")
    print("  " + "-" * 40)
    print(f"  {'Missed entirely (FN)':<25}  {missed_ent:>6}  {missed_ent/max(total_errors,1)*100:>5.0f}%")
    print(f"  {'Hallucinated (FP)':<25}  {hallucinated:>6}  {hallucinated/max(total_errors,1)*100:>5.0f}%")
    print(f"  {'Wrong entity type':<25}  {wrong_type:>6}  {wrong_type/max(total_errors,1)*100:>5.0f}%")
    print(f"  {'Boundary error':<25}  {boundary_err:>6}  {boundary_err/max(total_errors,1)*100:>5.0f}%")
    print(f"  {'TOTAL':<25}  {total_errors:>6}")

    # ── 6. RECALL vs PRECISION SPLIT ─────────────────────────────────
    print("\n" + "=" * 60)
    print("6. RECALL vs PRECISION ANALYSIS")
    print("=" * 60)

    low_recall_only  = 0  # R < 0.5, P >= 0.7 → model misses entities
    low_prec_only    = 0  # P < 0.5, R >= 0.7 → model hallucinates
    both_low         = 0  # Both bad

    for r in records:
        gold = parse_inline_tagged_output(r["gold_tagged"])
        pred = parse_inline_tagged_output(r["pred_tagged"])
        m = compute_ner_metrics([gold], [pred])
        o = m["overall"]
        p, rec = o["precision"], o["recall"]

        if rec < 0.5 and p >= 0.7:    low_recall_only += 1
        elif p < 0.5 and rec >= 0.7:  low_prec_only   += 1
        elif p < 0.5 and rec < 0.5:   both_low        += 1

    print(f"  Low recall only  (model MISSES entities):       {low_recall_only}")
    print(f"  Low precision only (model HALLUCINATES):        {low_prec_only}")
    print(f"  Both low (catastrophic failure):                {both_low}")

    # ── 7. WORST CHUNKS ───────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("7. WORST 10 CHUNKS")
    print("=" * 60)

    worst = sorted(records, key=lambda x: x["chunk_f1"])[:10]
    print(f"  {'#':<4}  {'Name':<30}  {'Sub':<10}  {'F1':>5}  {'Gold':>5}  {'Pred':>5}  {'TP':>4}")
    print("  " + "-" * 72)
    for r in worst:
        print(f"  {r['chunk_idx']:<4}  {r['chunk_name']:<30}  {r['subcorpus']:<10}  "
              f"{r['chunk_f1']:>5.3f}  {r['gold_count']:>5}  {r['pred_count']:>5}  {r['tp']:>4}")

    # ── 8. CHUNK SIZE CORRELATION ─────────────────────────────────────
    print("\n" + "=" * 60)
    print("8. CHUNK SIZE vs F1 CORRELATION")
    print("=" * 60)

    size_buckets = {"tiny (<50 words)": [], "small (50-100)": [],
                    "medium (100-200)": [], "large (>200)": []}

    for r in records:
        n_words = len(r["input_text"].split())
        f1 = r["chunk_f1"]
        if n_words < 50:         size_buckets["tiny (<50 words)"].append(f1)
        elif n_words < 100:      size_buckets["small (50-100)"].append(f1)
        elif n_words < 200:      size_buckets["medium (100-200)"].append(f1)
        else:                    size_buckets["large (>200)"].append(f1)

    print(f"  {'Size bucket':<20}  {'Count':>6}  {'Avg F1':>7}")
    print("  " + "-" * 37)
    for bucket, f1s in size_buckets.items():
        if f1s:
            print(f"  {bucket:<20}  {len(f1s):>6}  {sum(f1s)/len(f1s):>7.3f}")

    # ── 9. ENTITY DENSITY ─────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("9. ENTITY DENSITY vs DIFFICULTY")
    print("=" * 60)

    for r in records:
        n_words  = max(len(r["input_text"].split()), 1)
        density  = r["gold_count"] / n_words * 100
        print(f"  Chunk {r['chunk_idx']:>4} | gold={r['gold_count']:>3} | "
              f"words={n_words:>4} | density={density:>5.1f}% | F1={r['chunk_f1']:.3f}")

    # ── 10. SUMMARY ───────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("10. SUMMARY FOR THESIS")
    print("=" * 60)

    avg_f1   = sum(r["chunk_f1"] for r in records) / len(records)
    zero_f1  = sum(1 for r in records if r["chunk_f1"] == 0.0)
    top_sub  = max(sub_stats, key=lambda s: sub_stats[s]["count"])

    print(f"  Total hard chunks:        {len(records)}")
    print(f"  Avg F1 in hard chunks:    {avg_f1:.3f}")
    print(f"  Chunks with F1=0:         {zero_f1}")
    print(f"  Most problematic subcorp: {top_sub}")
    print(f"  Most missed type:         {fn_types.most_common(1)[0] if fn_types else 'N/A'}")
    print(f"  Most hallucinated type:   {fp_types.most_common(1)[0] if fp_types else 'N/A'}")

    return records


# Run it:
records = analyze_hard_chunks("hard_chunks_mamay9b_epoch4.jsonl")

Total hard chunks (F1 < threshold): 96
Subcorpora present: ['bruk', 'ng']

1. F1 DISTRIBUTION
  0.0 (zero)  :  19  ███████████████████
  0.0–0.3     :   5  █████
  0.3–0.5     :  13  █████████████
  0.5–0.7     :  59  ███████████████████████████████████████████████████████████

2. HARD CHUNKS PER SUBCORPUS
  Subcorpus        Hard chunks   Avg F1   Zero F1
  ---------------------------------------------
  bruk                      90    0.437        19
  ng                         6    0.604         0

3. ENTITY TYPE ERRORS (from per_type in hard chunks)
  Type       Seen    F1=0   F1<0.5   Zero%
  ----------------------------------------
  PERS         64      11       13     17%
  JOB          45      35       38     78%
  MISC         40      34       34     85%
  LOC          31       9       12     29%
  DATE         31       8        8     26%
  ORG          28       7        9     25%
  ART          26      16       17     62%
  PERIOD       19      10       10     53%
  DOC     

# Chunks for GEPA

In [ ]:
import json
import random
from collections import Counter, defaultdict

# Типи, яким гарантуємо мінімальне представлення у вибірці
RARE_TYPES  = {"JOB", "TIME", "DOC", "MISC", "PCT", "QUANT", "ART", "PERIOD"}

# Мінімум чанків де цей тип присутній
MIN_PER_RARE_TRAIN = 6
MIN_PER_RARE_VAL   = 2

# Мінімум "різноманітних" чанків (≥ 3 різних типів в одному чанку)
MIN_DIVERSE_TRAIN  = 24
MIN_DIVERSE_VAL    = 8

In [ ]:
def enrich_chunks_with_type_stats(chunks):
    """
    Додає до кожного чанку поля:
      type_counts   — Counter типів сутностей у цьому чанку
      n_types       — кількість унікальних типів
      rare_present  — множина рідкісних типів що є в чанку
      gold_entities — список (type, text) для метрики GEPA
    """
    enriched = []
    for chunk in chunks:
        # Отримуємо gold entities з tagged_text через вже існуючий парсер
        gold = parse_inline_tagged_output(chunk["tagged_text"])

        type_counts = Counter(t for t, _ in gold)
        rare_present = set(type_counts.keys()) & RARE_TYPES

        enriched.append({
            **chunk,
            "type_counts":   type_counts,
            "n_types":       len(type_counts),
            "rare_present":  rare_present,
            "gold_entities": gold,           # [(type, text), ...]
        })
    return enriched

In [ ]:
def stratified_sample(chunks, n_total, is_train, seed=42):
    """
    Стратифікована вибірка чанків.

    Алгоритм (3 кроки):
      1. Для кожного рідкісного типу — гарантуємо мінімум чанків.
      2. Добираємо різноманітні чанки (≥3 типів).
      3. Решту місць заповнюємо:
           train → пріоритет чанкам з рідкісними типами
           val   → рандомно (репрезентативність)
    """
    rng = random.Random(seed)
    min_rare    = MIN_PER_RARE_TRAIN if is_train else MIN_PER_RARE_VAL
    min_diverse = MIN_DIVERSE_TRAIN  if is_train else MIN_DIVERSE_VAL

    # Виключаємо порожні чанки
    pool = [c for c in chunks if c["num_entities"] > 0]
    rng.shuffle(pool)

    selected_names = set()
    selected = []

    def add(chunk):
        if chunk["name"] not in selected_names:
            selected_names.add(chunk["name"])
            selected.append(chunk)

    # ── Крок 1: рідкісні типи ──────────────────────────────────
    for rtype in RARE_TYPES:
        with_type = [c for c in pool if rtype in c["rare_present"]]
        if not with_type:
            print(f"[WARN] Тип {rtype} не знайдено в жодному чанку!")
            continue

        # Сортуємо: більше цього типу → більше різних типів → вперед
        with_type.sort(key=lambda c: (-c["type_counts"][rtype], -c["n_types"]))

        added = 0
        for chunk in with_type:
            if added >= min_rare:
                break
            if chunk["name"] not in selected_names:
                add(chunk)
                added += 1

        if added < min_rare:
            print(f"[WARN] {rtype}: вибрано {added}/{min_rare} чанків")

    # ── Крок 2: різноманітні чанки ────────────────────────────
    diverse = [c for c in pool if c["n_types"] >= 3 and c["name"] not in selected_names]
    diverse.sort(key=lambda c: -c["n_types"])

    cur_diverse = sum(1 for c in selected if c["n_types"] >= 3)
    for chunk in diverse:
        if cur_diverse >= min_diverse or len(selected) >= n_total:
            break
        add(chunk)
        cur_diverse += 1

    # ── Крок 3: заповнити решту ───────────────────────────────
    remaining = n_total - len(selected)
    if remaining > 0:
        leftover = [c for c in pool if c["name"] not in selected_names]

        if is_train:
            # Перевага чанкам де є хоч один рідкісний тип
            leftover.sort(key=lambda c: (-len(c["rare_present"]), -c["n_types"]))
        else:
            rng.shuffle(leftover)  # val — рандомно для репрезентативності

        for chunk in leftover[:remaining]:
            add(chunk)

    if len(selected) < n_total:
        print(f"[WARN] Вибрано {len(selected)} замість {n_total}")

    rng.shuffle(selected)
    return selected

In [ ]:
def to_gepa_example(chunk):
    """
    Конвертує збагачений чанк у словник для GEPA:
      input_text      — чистий текст (вхід моделі)
      expected_output — tagged_text (правильна відповідь)
      gold_entities   — [(type, text), ...] для метрики
    """
    return {
        "input_text":      chunk["input_text"],
        "expected_output": chunk["tagged_text"],
        "gold_entities":   chunk["gold_entities"],
        # Метадані (для відлагодження, GEPA їх ігнорує)
        "chunk_name":      chunk["name"],
        "subcorpus":       chunk["subcorpus"],
        "type_counts":     dict(chunk["type_counts"]),
    }


In [ ]:
def print_sample_stats(label, chunks, all_chunks):
    total_ents = sum(c["num_entities"] for c in chunks)
    type_counts = Counter()
    for c in chunks:
        type_counts.update(c["type_counts"])

    # Частота типів у повному датасеті (для порівняння)
    all_type_counts = Counter()
    for c in all_chunks:
        all_type_counts.update(c["type_counts"])
    all_total = sum(all_type_counts.values())

    print(f"\n{'='*60}")
    print(f"  {label}  ({len(chunks)} чанків, {total_ents} сутностей)")
    print(f"  Середня к-сть типів на чанк: "
          f"{sum(c['n_types'] for c in chunks)/len(chunks):.1f}")
    print(f"  Чанків з ≥3 типами: {sum(1 for c in chunks if c['n_types'] >= 3)}")
    print(f"\n  {'Тип':<8} | {'Sample':>8} | {'Dataset':>9} | {'Чанків':>7}")
    print(f"  {'-'*40}")

    sample_total = sum(type_counts.values()) or 1
    for etype in sorted(KEEP_ENTITIES):
        s_count = type_counts.get(etype, 0)
        d_freq  = all_type_counts.get(etype, 0) / all_total if all_total else 0
        n_chunks = sum(1 for c in chunks if etype in c["type_counts"])
        rare_tag = " *" if etype in RARE_TYPES else ""
        print(
            f"  {etype:<8} | {s_count/sample_total:>7.1%} | {d_freq:>8.1%} | "
            f"{n_chunks:>6}{rare_tag}"
        )
    print(f"  (* — рідкісний тип)")

In [ ]:
def build_gepa_dataset(
    train_chunks,
    test_chunks,
    n_train=50,
    n_val=20,
    seed=42,
    save_path=None,       # якщо вказано — зберігає JSON
    verbose=True,
):
    """
    Повний пайплайн: збагачення → вибірка → конвертація.

    Trainset береться з train_chunks (split="dev").
    Valset береться з test_chunks  (split="test").

    Повертає (trainset, valset) — списки словників для gepa.optimize().
    """
    print("[1/4] Збагачення чанків статистикою...")
    rich_train = enrich_chunks_with_type_stats(train_chunks)
    rich_test  = enrich_chunks_with_type_stats(test_chunks)

    print(f"[2/4] Вибірка: {n_train} train / {n_val} val...")
    train_sample = stratified_sample(rich_train, n_train, is_train=True,  seed=seed)
    val_sample   = stratified_sample(rich_test,  n_val,   is_train=False, seed=seed+1)

    if verbose:
        all_rich = rich_train + rich_test
        print_sample_stats("TRAIN SAMPLE", train_sample, all_rich)
        print_sample_stats("VAL SAMPLE",   val_sample,   all_rich)

    print("\n[3/4] Конвертація у формат GEPA...")
    trainset = [to_gepa_example(c) for c in train_sample]
    valset   = [to_gepa_example(c) for c in val_sample]

    if save_path:
        print(f"[4/4] Збереження у {save_path}...")
        os.makedirs(save_path, exist_ok=True)
        with open(f"{save_path}/gepa_trainset.json", "w", encoding="utf-8") as f:
            json.dump(trainset, f, ensure_ascii=False, indent=2)
        with open(f"{save_path}/gepa_valset.json", "w", encoding="utf-8") as f:
            json.dump(valset, f, ensure_ascii=False, indent=2)
        print(f"  Збережено: gepa_trainset.json ({len(trainset)} прикл.)")
        print(f"  Збережено: gepa_valset.json   ({len(valset)} прикл.)")
    else:
        print("[4/4] Збереження пропущено (save_path не вказано)")

    print(f"\n[DONE] trainset={len(trainset)}, valset={len(valset)}")
    return trainset, valset

In [ ]:
gepa_trainset, gepa_valset = build_gepa_dataset(
    train_chunks=train_chunks,
    test_chunks=test_chunks,
    n_train=100,
    n_val=40,
    seed=42,
    save_path="/content/gepa_data",   # або None
)

# Перевірка першого прикладу:
ex = gepa_trainset[0]
print("INPUT:", ex["input_text"])
print("GOLD:", ex["gold_entities"])
print("TYPES:", ex["type_counts"])

[1/4] Збагачення чанків статистикою...
[2/4] Вибірка: 100 train / 40 val...

  TRAIN SAMPLE  (100 чанків, 2034 сутностей)
  Середня к-сть типів на чанк: 6.6
  Чанків з ≥3 типами: 95

  Тип      |   Sample |   Dataset |  Чанків
  ----------------------------------------
  ART      |    6.4% |     2.8% |     59 *
  DATE     |    8.2% |     8.0% |     71
  DOC      |    1.8% |     0.6% |     23 *
  JOB      |   10.9% |     9.2% |     77 *
  LOC      |   10.2% |    13.8% |     62
  MISC     |    3.6% |     2.4% |     27 *
  MON      |    5.9% |     4.4% |     52
  ORG      |   27.6% |    23.8% |     89
  PCT      |    3.2% |     1.2% |     27 *
  PERIOD   |    4.2% |     2.7% |     55 *
  PERS     |   12.9% |    29.1% |     72
  QUANT    |    4.5% |     1.8% |     38 *
  TIME     |    0.5% |     0.2% |      7 *
  (* — рідкісний тип)

  VAL SAMPLE  (40 чанків, 609 сутностей)
  Середня к-сть типів на чанк: 4.8
  Чанків з ≥3 типами: 28

  Тип      |   Sample |   Dataset |  Чанків
  ----------

# GEPA Metrics

In [ ]:
from collections import Counter

def gepa_ner_metric(example: dict, prediction: str) -> dict:
    """
    Метрика для GEPA. Приймає один приклад і відповідь моделі.
    Повертає {"score": f1, "feedback": "..."}.

    example["gold_entities"] — список [(type, text), ...] з gepa_trainset
    prediction               — сирий вивід моделі (inline-tagged текст)
    """
    gold_entities = example["gold_entities"]
    pred_entities = parse_inline_tagged_output(prediction)

    # Нормалізація через твою існуючу функцію
    gold_norm = [(t, normalize_evaluation_text(txt)) for t, txt in gold_entities]
    pred_norm = [(t, normalize_evaluation_text(txt)) for t, txt in pred_entities]

    gold_counts = Counter(gold_norm)
    pred_counts = Counter(pred_norm)

    # TP / FP / FN
    tp = sum((gold_counts & pred_counts).values())
    fp = sum((pred_counts - gold_counts).values())
    fn = sum((gold_counts - pred_counts).values())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # ── Текстовий feedback для reflection LLM ──────────────────

    missed       = list((gold_counts - pred_counts).elements())
    hallucinated = list((pred_counts - gold_counts).elements())

    # Per-type статистика (тільки для типів де є помилки)
    type_errors = {}
    for (etype, _) in missed:
        type_errors.setdefault(etype, {"fn": 0, "fp": 0})
        type_errors[etype]["fn"] += 1
    for (etype, _) in hallucinated:
        type_errors.setdefault(etype, {"fn": 0, "fp": 0})
        type_errors[etype]["fp"] += 1

    lines = [f"F1={f1:.3f} (P={precision:.3f}, R={recall:.3f}) | "
             f"TP={tp}, FP={fp}, FN={fn}"]

    if not missed and not hallucinated:
        lines.append("Всі сутності визначено правильно.")
    else:
        if missed:
            # Групуємо пропущені по типу — компактніше для reflection LLM
            by_type = {}
            for etype, etext in missed:
                by_type.setdefault(etype, []).append(etext)
            parts = []
            for etype, texts in sorted(by_type.items()):
                examples = ", ".join(f'"{t}"' for t in texts)
                parts.append(f"{etype}: {examples}")
            lines.append(f"ПРОПУЩЕНО ({fn}): {' | '.join(parts)}")

        if hallucinated:
            by_type = {}
            for etype, etext in hallucinated:
                by_type.setdefault(etype, []).append(etext)
            parts = []
            for etype, texts in sorted(by_type.items()):
                examples = ", ".join(f'"{t}"' for t in texts)
                parts.append(f"{etype}: {examples}")
            lines.append(f"ЗАЙВЕ/ГАЛЮЦИНАЦІЇ ({fp}): {' | '.join(parts)}")

        # Резюме по типах де є проблеми
        if type_errors:
            type_summary = []
            for etype, errs in sorted(type_errors.items()):
                fn_str = f"пропущено {errs['fn']}" if errs["fn"] else ""
                fp_str = f"зайве {errs['fp']}"     if errs["fp"] else ""
                detail = ", ".join(filter(None, [fn_str, fp_str]))
                type_summary.append(f"{etype}({detail})")
            lines.append(f"Проблемні типи: {', '.join(type_summary)}")

    feedback = "\n".join(lines)
    return {"score": f1, "feedback": feedback}

In [ ]:
# Тест 1: ідеальна відповідь
ex = gepa_trainset[0]
result = gepa_ner_metric(ex, ex["expected_output"])
print("=== Ідеальна відповідь ===")
print(f"Score: {result['score']:.3f}")
print(result["feedback"])

# Тест 2: порожня відповідь
result = gepa_ner_metric(ex, ex["input_text"])  # текст без тегів
print("\n=== Порожня відповідь ===")
print(f"Score: {result['score']:.3f}")
print(result["feedback"])

=== Ідеальна відповідь ===
Score: 1.000
F1=1.000 (P=1.000, R=1.000) | TP=26, FP=0, FN=0
Всі сутності визначено правильно.

=== Порожня відповідь ===
Score: 0.000
F1=0.000 (P=0.000, R=0.000) | TP=0, FP=0, FN=26
ПРОПУЩЕНО (26): DATE: "2015 року" | JOB: "засновників", "Начальник обласного управління лісового та мисливського господарства", "Начальник обласного управління лісового та мисливського господарства", "депутатом", "Солідарність БПП", "директорів", "керівника обласного управління лісового та мисливського господарства", "міністра", "керівників" | LOC: "Рівненській області" | ORG: "Млинівського держлісгоспу", "МГ Лісовик- Гранд", "Млинівський держлісгосп", "Лісовик- Гранд", "Наших грошей", "Рівненської обласної ради", "ДП Висоцький лісгосп", "Рівненського обласного управління лісового та мисливського господарства", "БПП" | PERS: "Віталій Сухович", "Віталій Сухович", "Сухович", "Сухович", "Арсена Авакова" | QUANT: "4,25 тис гектарів"
Проблемні типи: DATE(пропущено 1), JOB(пропущено 9)

# GEPA Adapter

In [ ]:
from gepa.core.adapter import GEPAAdapter, EvaluationBatch
from collections.abc import Mapping, Sequence
from typing import Any
import random

class NERGEPAAdapter:
    """
    GEPA adapter для локальних NER моделей (Gemma-2/3 family).
    Сумісний з поточним GEPA API (batch, candidate, capture_traces).
    """

    propose_new_texts = None

    def __init__(self, model, tokenizer, batch_size: int = 4, seed: int = 42):
        self.model = model
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self._rng = random.Random(seed)

    # ──────────────────────────────────────────────────────────────
    # 1. EVALUATE
    # Нова сигнатура: (batch, candidate, capture_traces=False)
    # Повертає EvaluationBatch, а не list[dict]
    # ──────────────────────────────────────────────────────────────
    def evaluate(
        self,
        batch: list[dict],
        candidate: dict[str, str],
        capture_traces: bool = False,
    ) -> EvaluationBatch:

        system_prompt = candidate["system_prompt"]
        texts = [ex["input_text"] for ex in batch]

        # ── Batched inference ──────────────────────────────────────
        responses = []
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i : i + self.batch_size]

            prompts = []
            for text in batch_texts:
              messages = create_chat_messages(text, system_prompt=system_prompt)
              prompt = self.tokenizer.apply_chat_template(
                  messages,
                  tokenize=False,
                  add_generation_prompt=True,
              )
              prompts.append(prompt)

            inputs = self.tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048,
            ).to(self.model.device)

            longest = max(batch_texts, key=len)
            text_tokens = len(
                self.tokenizer(longest, add_special_tokens=False).input_ids
            )
            max_new_tokens = int(text_tokens * 1.3) + 50

            import torch
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    temperature=None,
                    top_p=None,
                    top_k=None,
                    repetition_penalty=1.0,
                    eos_token_id=self.tokenizer.eos_token_id,
                    pad_token_id=self.tokenizer.pad_token_id,
                )

            padded_len = inputs.input_ids.shape[1]
            for output in outputs:
                response = self.tokenizer.decode(
                    output[padded_len:], skip_special_tokens=True
                )
                responses.append(response)

        # ── Score each example ─────────────────────────────────────
        raw_outputs = []
        scores = []
        trajectories = [] if capture_traces else None

        for ex, response in zip(batch, responses):
            metric_result = gepa_ner_metric(ex, response)
            score = metric_result["score"]
            scores.append(score)
            raw_outputs.append(response)

            if capture_traces:
                # trajectory = everything the reflection LM will need later
                trajectories.append({
                    "input_text":     ex["input_text"],
                    "response":       response,
                    "score":          score,
                    "feedback":       metric_result["feedback"],
                    "gold_entities":  ex["gold_entities"],
                })

        return EvaluationBatch(
            outputs=raw_outputs,
            scores=scores,
            trajectories=trajectories,  # None if capture_traces=False
        )

    # ──────────────────────────────────────────────────────────────
    # 2. MAKE_REFLECTIVE_DATASET
    # Замінює старий extract_traces_for_reflection
    # Повертає dict: component_name -> list of reflection records
    # ──────────────────────────────────────────────────────────────
    def make_reflective_dataset(
        self,
        candidate: dict[str, str],
        eval_batch: EvaluationBatch,
        components_to_update: list[str],
    ) -> Mapping[str, Sequence[Mapping[str, Any]]]:

        trajectories = eval_batch.trajectories or []

        # Sort by score — worst examples first (most signal for reflection)
        sorted_traj = sorted(trajectories, key=lambda t: t["score"])

        # Take up to 5 worst examples (don't overflow reflection LM context)
        worst = sorted_traj[:5]

        records = []
        for traj in worst:
            records.append({
                "Inputs": {
                    "text": traj["input_text"][:500],  # truncate for context
                },
                "Generated Outputs": {
                    "tagged_text": traj["response"][:700],
                },
                "Feedback": traj["feedback"],
                # Extra info — helps reflection LM understand the task
                "score": round(traj["score"], 3),
                "gold_entities": traj["gold_entities"][:15],  # first 15 gold
            })

        # Return same records for every component being updated
        # (we only have one component: "system_prompt")
        return {comp: records for comp in components_to_update}

# Run GEPA

In [ ]:
import os
import json
import gepa
from gepa.core.adapter import GEPAAdapter, EvaluationBatch

def run_gepa_optimization(
    model,
    tokenizer,
    trainset: list[dict],
    valset: list[dict],
    model_name: str,          # для збереження результатів, напр. "mamaylm_9b"
    save_dir: str,            # куди зберігати оптимізовані промпти
    max_metric_calls: int = 100,
    batch_size: int = 4,
    google_api_key: str = None,
) -> dict:
    """
    Запускає GEPA оптимізацію промпту для вказаної моделі.
    Повертає словник з результатами.
    """
    if google_api_key:
        os.environ["GEMINI_API_KEY"] = google_api_key

    adapter = NERGEPAAdapter(model, tokenizer, batch_size=batch_size)

    seed_prompt = {"system_prompt": SYSTEM_PROMPT}

    print(f"\n{'='*60}")
    print(f"  GEPA оптимізація: {model_name}")
    print(f"  max_metric_calls={max_metric_calls} | "
          f"train={len(trainset)} | val={len(valset)}")
    print(f"{'='*60}\n")

    result = gepa.optimize(
        adapter=adapter,
        seed_candidate=seed_prompt,
        trainset=trainset,
        valset=valset,
        reflection_lm="gemini/gemma-4-31b-it",
        max_metric_calls=max_metric_calls,
        # metric=gepa_ner_metric,
    )


    # Збереження результатів
    output = {
        "model_name":       model_name,
        "best_score":       result.val_aggregate_scores[result.best_idx],
        "best_prompt":      result.best_candidate["system_prompt"],
        "seed_prompt":      SYSTEM_PROMPT,
        "max_metric_calls": max_metric_calls,
        "n_train":          len(trainset),
        "n_val":            len(valset),
    }

    save_path = os.path.join(save_dir, f"gepa_result_{model_name}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"\n[DONE] {model_name}")
    print(f"  Seed score:  {_eval_seed_score(trainset[:5]):.3f}")
    print(f"  Best score:  {result.val_aggregate_scores[result.best_idx]:.3f}")
    print(f"  Збережено:   {save_path}")
    print(f"\nОптимізований промпт:\n{result.best_candidate['system_prompt']}")

    return output


def _eval_seed_score(examples: list[dict]) -> float:
    """Швидка оцінка seed промпту на кількох прикладах для порівняння."""
    scores = [gepa_ner_metric(ex, ex["expected_output"])["score"] for ex in examples]
    return sum(scores) / len(scores) if scores else 0.0


def load_gepa_result(model_name: str, save_dir: str) -> dict:
    """Завантажує збережений результат GEPA."""
    path = os.path.join(save_dir, f"gepa_result_{model_name}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def print_gepa_comparison(save_dir: str, model_names: list[str]):
    """Виводить порівняльну таблицю результатів по всіх моделях."""
    print(f"\n{'='*55}")
    print(f"  {'Модель':<20} | {'Seed':>6} | {'Best':>6} | {'Приріст':>8}")
    print(f"  {'-'*50}")
    for name in model_names:
        try:
            r = load_gepa_result(name, save_dir)
            # seed score рахуємо окремо якщо збережено, інакше N/A
            best = r["best_score"]
            print(f"  {name:<20} | {'N/A':>6} | {best:>6.3f} | {'—':>8}")
        except FileNotFoundError:
            print(f"  {name:<20} | {'—':>6} | {'—':>6} | не запущено")
    print(f"{'='*55}")

In [ ]:
from google.colab import userdata

GEPA_SAVE_DIR  = "/content/gepa_results"
GOOGLE_API_KEY = userdata.get("GEMINI_API_KEY")

# # ──────────────────────────────────────────────────────
# # ТЕСТ — швидка перевірка що все працює (~5 хвилин)
# # 10 metric calls, 5 train прикладів, 3 val
# # Не для реальної оптимізації — тільки щоб перевірити
# # що адаптер, метрика і GEPA підключились без помилок
# # ──────────────────────────────────────────────────────
# result_test = run_gepa_optimization(
#     model=model,
#     tokenizer=tokenizer,
#     trainset=gepa_trainset[:5],
#     valset=gepa_valset[:3],
#     model_name="mamaylm_9b_TEST",
#     save_dir=GEPA_SAVE_DIR,
#     max_metric_calls=10,
#     batch_size=16,
#     google_api_key=GOOGLE_API_KEY,
# )

In [ ]:
# ──────────────────────────────────────────────────────
# ПОВНИЙ ЗАПУСК — реальна оптимізація (~2-3 години)
# 100 metric calls, повний trainset/valset
# ──────────────────────────────────────────────────────
GEPA_SAVE_DIR  = "/content/gepa_results"

result_full = run_gepa_optimization(
    model=model,
    tokenizer=tokenizer,
    trainset=gepa_trainset,
    valset=gepa_valset,
    model_name="mamaylm_9b",
    save_dir=GEPA_SAVE_DIR,
    max_metric_calls=500,
    batch_size=12,
    google_api_key=GOOGLE_API_KEY,
)


  GEPA оптимізація: mamaylm_9b
  max_metric_calls=500 | train=100 | val=40

Iteration 0: Base program full valset score: 0.4474260604475713 over 40 / 40 examples
Iteration 1: Selected program 0 score: 0.4474260604475713
Iteration 1: Proposed new text for system_prompt: Ти — інструмент прецизійної розмітки тексту (Named Entity Recognition), а не редактор. Твоє завдання — ідентифікувати іменовані сутності в українському тексті та позначити їх у форматі [ТИП: сутність].

### СЛОВНИК ТИПІВ СУТНОСТЕЙ:
- **PERS** — імена людей, прізвища, літературні персонажі та міфічні істоти.
- **ORG** — конкретні організації, компанії, установи, партії, проєкти, конференції. (Увага: не позначай як ORG прикметники національності, наприклад, "німецьких").
- **LOC** — географічні назви: райони, міста, країни, ріки, озера, моря, гори, а також повні адреси (наприклад, "вул. Ювілейна, 60А").
- **DATE** — повна або часткова календарна дата (століття, рік, місяць, день).
- **TIME** — текстова або числова мітка ч